In [1]:
# ══════════════════════════════════════════════════════════════
# SOL/USD — Advanced Model Factory v4.1
# ══════════════════════════════════════════════════════════════
import requests
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.feature_selection import SelectKBest, f_classif
import joblib
import os
import json
import random
from datetime import datetime, timedelta
import time
import logging
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

# ══════════════════════════════════════════════════════════════
# ★ المتغيرات الأساسية (SOL)
# ══════════════════════════════════════════════════════════════
TWELVE_DATA_API_KEY = os.environ.get("TWELVE_DATA_API_KEY", "")
SYMBOL       = "SOL/USD"
SYMBOL_SAFE  = "SOLUSD"
CSV_RAW_FILE = f"{SYMBOL_SAFE}_5min_raw.csv"
MODELS_DIR   = os.path.join("models", SYMBOL_SAFE)

# ── إدارة المخاطر (SOL أعلى تقلب — أشد تحفظاً من ETH) ──
RISK_PCT_PER_TRADE = 0.012   # ⬇️ من 0.015 (ETH) — SOL vol ~1.5-2x ETH
MAX_DAILY_RISK_PCT = 0.04    # ⬇️ من 0.05
INITIAL_CAPITAL    = 10_000
SPREAD_BPS         = 5       # ⬆️ من 3 — SOL spread أوسع
SLIPPAGE           = 0.0009  # ⬆️ من 0.0006 — سيولة SOL أقل من ETH

# ── Confidence-sizing (Kelly-lite) ─────────────────
USE_CONFIDENCE_SIZING = True
MIN_SIZE_FRACTION     = 0.4    # ⬇️ من 0.5 — تحفظ أكتر مع SOL
MAX_SIZE_FRACTION     = 1.4    # ⬇️ من 1.5 — cap أشد

# ── ★ v4.3: Signal Quality Gates ──────────────────
MIN_PROB_MARGIN         = 0.02
MIN_COST_COVERAGE       = 2.5
USE_SQUEEZE_FILTER      = True
MAX_CONSECUTIVE_LOSSES  = 5
COOLDOWN_BARS_MULT      = 1.0

# ── ★ v4.4: Recency weighting for XGB training ────
# SOL بيتطور بسرعة — آخر البيانات أهم من الأقدم
USE_RECENCY_WEIGHTING   = True
RECENCY_WEIGHT_MIN      = 1.0     # oldest samples weight
RECENCY_WEIGHT_MAX      = 3.0     # newest samples weight

# ── ★ v4.4: Fold threshold optimizer control ──────
# الـ optimizer على fold صغير بيلاقي threshold "حظ" → ضرر أكتر من فايدة
DISABLE_FOLD_THRESHOLD_OPT = True   # WF folds use fold_buy_threshold directly

# ── ★ v4.4: 1W-specific overrides ─────────────────
FORCE_MAX_YEARS_1W      = 5       # SOL doesn't exist before 2020
HOLDOUT_ZERO_TRADE_RETRIES = 3    # if holdout=0 trades, retry with lower th
HOLDOUT_ZERO_TRADE_STEP    = 0.05 # each retry: buy_th -= 0.05
HOLDOUT_RETRY_FLOOR        = 0.20 # ★ v4.5: retry floor lowered 0.30→0.20

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("training_solusd_v4.log", encoding='utf-8'),
        logging.StreamHandler()
    ]
)
print(f"✅ Setup done for {SYMBOL} | Risk={RISK_PCT_PER_TRADE*100:.1f}% | "
      f"Spread={SPREAD_BPS}bps | Capital=${INITIAL_CAPITAL:,}")

✅ Setup done for SOL/USD | Risk=1.2% | Spread=5bps | Capital=$10,000


## إعدادات الفريمات — معايرة تجريبية لـ SOL

**فلسفة الضبط:**
- **SL أوسع بـ 0.3-0.5x ATR** عن ETH (SOL بيعمل fake-outs أكتر)
- **TP4 أعلى بـ 0.5-2x ATR** (SOL بيمتد أكتر في الترنداتالقوية)
- **Buy threshold أعلى بـ 0.02-0.03** (فلترة noise أعلى)
- **min_adx أعلى بـ 2 نقطة** على الفريمات السريعة
- **max_trades_per_day أقل** (تقليل over-trading في السوق المتقلب)
- **horizon أطول شوية على 5min/15min** (SOL momentum بيستغرق وقت)

In [11]:
# ══════════════════════════════════════════════════════════════
# TIMEFRAME CONFIG v4.1 SOL — معايَر لخصائص SOL/USD
# ══════════════════════════════════════════════════════════════
TIMEFRAME_CONFIG = {
    "5min": {
        "interval": "5min", "years": 2,
        # ★ v4.3: تشديد جوهري — كان بيولد 696 trade بـ WR 39.5% وخسارة 2.21%
        "buy_threshold":      0.55,      # ★ v4.5: 0.72→0.58 (كان بيخنق الإشارات)
        "sell_threshold":     0.42,      # ★ v4.5: 0.28→0.42
        "fold_buy_threshold": 0.54,      # ★ v4.5: 0.66→0.54
        "fold_sell_threshold":0.46,      # ★ v4.5: 0.34→0.46
        "max_trades_per_day": 2,         # ★ v4.5: 1→2
        "max_trades_per_fold":120,       # ★ v4.5: 50→120
        "min_prob_margin":    0.01,      # ★ v4.5: 0.03→0.01
        "min_adx_strict":     22,        # ★ v4.5: 28→22
        "resample_rule": "5min",
        "larger_tf": "1h",
        "htf_features": ["rsi","trend_filter","macd_hist",
                         "market_structure","adx","ema_alignment",
                         "vol_regime_htf","breakout_intensity_up",
                         "breakout_intensity_dn"],
        "minutes_per_candle": 5,
        "min_candles": 500,
        "horizon": 24,
        "atr_mult_sl":  3.6,
        "atr_mult_tp1": 2.2,             # ★ v4.3: 2.0→2.2 (better cost coverage)
        "atr_mult_tp2": 3.6,             # ★ v4.3: 3.4→3.6
        "atr_mult_tp3": 5.4,
        "atr_mult_tp4": 7.5,
        "min_trades_fold": 3,            # ★ v4.3: 5→3 (allow smaller folds)
        "min_adx": 20,                   # ★ v4.5: 28→20 (كان صارم جداً)
    },
    "15min": {
        "interval": "15min", "years": 3,
        # ★ v4.3: كان 896 trade، WR 40%، Return -4.31% — تشديد كامل
        "buy_threshold":      0.54,      # ★ v4.5: 0.70→0.57
        "sell_threshold":     0.43,      # ★ v4.5
        "fold_buy_threshold": 0.53,      # ★ v4.5: 0.65→0.54
        "fold_sell_threshold":0.46,      # ★ v4.5
        "max_trades_per_day": 2,         # ★ v4.5: 1→2
        "max_trades_per_fold":150,       # ★ v4.5: 60→150
        "min_prob_margin":    0.01,      # ★ v4.5: 0.025→0.01
        "min_adx_strict":     22,        # ★ v4.5: 26→22
        "resample_rule": "15min",
        "larger_tf": "4h",
        "htf_features": ["rsi","trend_filter","atr_pct",
                         "market_structure","adx","vol_regime_htf",
                         "breakout_intensity_up"],
        "minutes_per_candle": 15,
        "min_candles": 350,
        "horizon": 20,
        "atr_mult_sl":  3.6,
        "atr_mult_tp1": 2.4,             # ★ v4.3: 2.2→2.4
        "atr_mult_tp2": 4.0,             # ★ v4.3: 3.8→4.0
        "atr_mult_tp3": 5.8,
        "atr_mult_tp4": 8.2,
        "min_trades_fold": 3,            # ★ v4.3: 5→3
        "min_adx": 20,                   # ★ v4.5: 26→20
    },
    "45min": {
        "interval": "45min", "years": 5,
        # ★ v4.3: WF لسه ضعيف (Return -0.38%) لكن Holdout +4.93% — نشدد للـ WF
        "buy_threshold":      0.55,      # ★ v4.5: 0.66→0.56
        "sell_threshold":     0.44,      # ★ v4.5
        "fold_buy_threshold": 0.53,      # ★ v4.5: 0.62→0.53
        "fold_sell_threshold":0.47,      # ★ v4.5
        "max_trades_per_day": None,
        "max_trades_per_fold":100,       # ★ v4.5: 50→100
        "min_prob_margin":    0.01,      # ★ v4.5: 0.02→0.01
        "min_adx_strict":     20,        # ★ v4.5: 24→20
        "resample_rule": "45min",
        "larger_tf": "4h",
        "htf_features": ["rsi","trend_filter","macd_hist",
                         "market_structure","vol_regime_htf",
                         "breakout_intensity_up"],
        "minutes_per_candle": 45,
        "min_candles": 200,
        "horizon": 15,
        "atr_mult_sl":  3.8,
        "atr_mult_tp1": 2.8,             # ★ v4.3: 2.6→2.8
        "atr_mult_tp2": 4.6,             # ★ v4.3: 4.4→4.6
        "atr_mult_tp3": 6.4,
        "atr_mult_tp4": 9.0,
        "min_trades_fold": 3,            # ★ v4.3: 4→3
        "min_adx": 18,       # ★ v4.5: 24→18                   # ★ v4.3: 18→24
    },
    "1h": {
        "interval": "1h", "years": 6,
        # ★ v4.3: كان 1056 WF trade فقط 28 holdout — WF بيتاجر أكتر بكتير مما ينبغي
        "buy_threshold":      0.56,      # ★ v4.5: 0.68→0.56 (holdout أثبت 0.54 يدي +17.79%!)
        "sell_threshold":     0.44,      # ★ v4.5: 0.32→0.44
        "fold_buy_threshold": 0.54,      # ★ v4.5: 0.64→0.54
        "fold_sell_threshold":0.46,      # ★ v4.5: 0.36→0.46
        "max_trades_per_day": None,
        "max_trades_per_fold":120,       # ★ v4.5: 60→120
        "min_prob_margin":    0.01,      # ★ v4.5: 0.02→0.01
        "min_adx_strict":     20,        # ★ v4.5: 24→20
        "resample_rule": "1h",
        "larger_tf": "4h",
        "htf_features": ["rsi","trend_filter","market_structure",
                         "sr_dist","adx","ema_alignment","vol_regime_htf",
                         "breakout_intensity_up","momentum_persistence"],
        "minutes_per_candle": 60,
        "min_candles": 200,
        "horizon": 18,
        "atr_mult_sl":  4.0,
        "atr_mult_tp1": 2.9,             # ★ v4.3: 2.7→2.9
        "atr_mult_tp2": 4.8,             # ★ v4.3: 4.6→4.8
        "atr_mult_tp3": 6.8,
        "atr_mult_tp4": 10.0,
        "min_trades_fold": 3,            # ★ v4.3: 5→3
        "min_adx": 18,       # ★ v4.5: 24→18                   # ★ v4.3: 18→24
    },
    "2h": {
        "interval": "2h", "years": 6,
        "buy_threshold":      0.55,      # ⬆️ من ETH 0.60
        "sell_threshold":     0.39,
        "fold_buy_threshold": 0.53,      # ⬆️ من ETH 0.55
        "fold_sell_threshold":0.44,
        "max_trades_per_day": None,
        "resample_rule": "2h",
        "larger_tf": "1d",
        "htf_features": ["rsi","trend_filter","atr_pct",
                         "market_structure","vol_regime_htf",
                         "breakout_intensity_up"],
        "minutes_per_candle": 120,
        "min_candles": 150,
        "horizon": 18,
        "atr_mult_sl":  4.2,             # ⬆️ من ETH 4.0
        "atr_mult_tp1": 2.8,             # ⬆️ من 2.6
        "atr_mult_tp2": 4.8,             # ⬆️ من 4.5
        "atr_mult_tp3": 7.0,             # ⬆️ من 6.5
        "atr_mult_tp4": 11.0,            # ⬆️ من ETH 10.0
        "min_trades_fold": 4,
        "min_adx": 16,                   # ⬆️ من ETH 15
    },
    "4h": {
        "interval": "4h", "years": 5,    # ⬇️ من ETH 7 — SOL أحدث نسبياً
        "buy_threshold":      0.55,      # ⬆️ من ETH 0.56
        "sell_threshold":     0.43,
        "fold_buy_threshold": 0.53,      # ⬆️ من ETH 0.52
        "fold_sell_threshold":0.47,
        "max_trades_per_day": None,
        "resample_rule": "4h",
        "larger_tf": "1d",
        "htf_features": ["rsi","trend_filter","market_structure",
                         "sr_dist","adx","ema_alignment","vol_regime_htf",
                         "momentum_persistence"],
        "minutes_per_candle": 240,
        "min_candles": 100,
        "horizon": 18,
        "atr_mult_sl":  4.0,             # ⬆️ من ETH 3.8
        "atr_mult_tp1": 3.0,             # ⬆️ من 2.8
        "atr_mult_tp2": 4.8,             # ⬆️ من 4.5
        "atr_mult_tp3": 7.0,             # ⬆️ من 6.5
        "atr_mult_tp4": 11.0,            # ⬆️ من ETH 10.0
        "min_trades_fold": 3,
        "min_adx": 0,
    },
    "1d": {
        "interval": "1d", "years": 5,    # ⬇️ من ETH 8 — SOL launch 2020
        "buy_threshold":      0.53,      # ⬆️ من ETH 0.53
        "sell_threshold":     0.46,
        "fold_buy_threshold": 0.50,      # ⬆️ من ETH 0.48
        "fold_sell_threshold":0.51,
        "max_trades_per_day": None,
        "resample_rule": "1D",
        "larger_tf": "1W",
        "htf_features": ["rsi","trend_filter","market_structure","momentum_persistence"],
        "minutes_per_candle": 1440,
        "min_candles": 120,
        "horizon": 18,
        "atr_mult_sl":  4.0,             # ⬆️ من ETH 3.8
        "atr_mult_tp1": 3.2,             # ⬆️ من 3.0
        "atr_mult_tp2": 5.4,             # ⬆️ من 5.0
        "atr_mult_tp3": 8.0,             # ⬆️ من 7.5
        "atr_mult_tp4": 12.5,            # ⬆️ من ETH 11.5
        "min_trades_fold": 2,
        "min_adx": 0,
    },
    "1W": {
        # ★ v4.3: class imbalance في v4.2 كان 10.2% pos → WR 6.25% — نعيد تعريف الـ target
        # الحل: SL أوسع + TP1 أسهل + horizon أطول = target متوازن ~30%
        "interval": "1week", "years": 5,   # ★ v4.4: capped at 5 (SOL launched March 2020)
        "buy_threshold":      0.38,      # ★ v4.5: 0.42→0.38 (holdout retry kept hitting floor)
        "sell_threshold":     0.58,
        "fold_buy_threshold": 0.36,      # ★ v4.5: 0.40→0.36
        "fold_sell_threshold":0.60,
        "max_trades_per_day": None,
        "max_trades_per_fold":None,      # 1W: مفيش cap (trades قليلة أصلاً)
        "min_prob_margin":    0.0,       # 1W: no extra margin needed
        "min_adx_strict":     0,         # 1W: مفيش ADX filter
        "smote_multiplier":   2.0,       # ★ v4.3 NEW: ×2 scale_pos_weight for 1W
        "resample_rule": "W-MON",
        "larger_tf": None,
        "htf_features": [],
        "minutes_per_candle": 10080,
        "min_candles": 20,
        "horizon": 6,                    # ★ v4.3: 5→6 (وقت أطول للـ TP1 يحقق)
        "atr_mult_sl":  6.0,             # ★ v4.3: 4.2→6.0 (weekly ATR كبير — space)
        "atr_mult_tp1": 2.0,             # ★ v4.3: 2.5→2.0 (easier target)
        "atr_mult_tp2": 3.5,             # ★ v4.3: 4.5→3.5
        "atr_mult_tp3": 5.5,             # ★ v4.3: 6.5→5.5
        "atr_mult_tp4": 8.0,             # ★ v4.3: 10.0→8.0
        "min_trades_fold": 1,
        "min_adx": 0,
    },
}

# ★ v4.2 NEW: notional cap to prevent extreme leverage on low-vol candles
MAX_POSITION_PCT = 0.30   # cap position notional at 30% of capital
print(f"✅ Loaded {len(TIMEFRAME_CONFIG)} SOL-tuned timeframe configs")

✅ Loaded 8 SOL-tuned timeframe configs


In [12]:
# ══════════════════════════════════════════════════════════════
# XGBoost params — مخصصة لكل فريم (SOL: regularization أعلى شوية)
# ══════════════════════════════════════════════════════════════
XGB_PARAMS = {
    # الفريمات السريعة: SOL أعلى noise → regularization أشد
    "5min":  {"max_depth": 4, "learning_rate": 0.028, "subsample": 0.72,
              "colsample_bytree": 0.62, "reg_alpha": 2.2, "reg_lambda": 4.0,
              "min_child_weight": 14, "gamma": 0.7},
    "15min": {"max_depth": 4, "learning_rate": 0.032, "subsample": 0.72,
              "colsample_bytree": 0.62, "reg_alpha": 1.8, "reg_lambda": 3.5,
              "min_child_weight": 12, "gamma": 0.6},
    "45min": {"max_depth": 5, "learning_rate": 0.032, "subsample": 0.75,
              "colsample_bytree": 0.68, "reg_alpha": 1.5, "reg_lambda": 3.0,
              "min_child_weight": 10, "gamma": 0.5},
    "1h":    {"max_depth": 5, "learning_rate": 0.038, "subsample": 0.78,
              "colsample_bytree": 0.68, "reg_alpha": 1.2, "reg_lambda": 2.5,
              "min_child_weight": 8,  "gamma": 0.4},
    "2h":    {"max_depth": 5, "learning_rate": 0.040, "subsample": 0.78,
              "colsample_bytree": 0.70, "reg_alpha": 1.0, "reg_lambda": 2.0,
              "min_child_weight": 6,  "gamma": 0.3},
    "4h":    {"max_depth": 6, "learning_rate": 0.042, "subsample": 0.80,
              "colsample_bytree": 0.72, "reg_alpha": 0.6, "reg_lambda": 1.2,
              "min_child_weight": 5,  "gamma": 0.25},
    "1d":    {"max_depth": 6, "learning_rate": 0.048, "subsample": 0.82,
              "colsample_bytree": 0.75, "reg_alpha": 0.4, "reg_lambda": 1.0,
              "min_child_weight": 4,  "gamma": 0.15},
    "1W":    {"max_depth": 4, "learning_rate": 0.055, "subsample": 0.85,
              "colsample_bytree": 0.80, "reg_alpha": 0.5, "reg_lambda": 1.0,
              "min_child_weight": 2,  "gamma": 0.0},
    "default": {"max_depth": 5, "learning_rate": 0.04, "subsample": 0.78,
                "colsample_bytree": 0.70, "reg_alpha": 1.2, "reg_lambda": 2.5,
                "min_child_weight": 8, "gamma": 0.4},
}

MAX_FEATURES = {
    "5min": 90, "15min": 90, "45min": 80, "1h": 80,
    "2h": 70, "4h": 70, "1d": 60, "1W": 45,
}

print("✅ SOL XGB params + feature caps loaded (v4.1)")

✅ SOL XGB params + feature caps loaded (v4.1)


##  Data Fetcher — TwelveData API

نفس الفَاتش تبع ETH، بس مضبوط على SOL/USD.
> **ملاحظة:** تحتاج API key من TwelveData. صنف مجاني بيدي 800 request/day.

In [13]:
# ══════════════════════════════════════════════════════════════
# DATA FETCHER (TwelveData API)
# ══════════════════════════════════════════════════════════════
class TwelveDataFetcher:
    BASE_URL = "https://api.twelvedata.com/time_series"

    def __init__(self, api_key: str):
        self.api_key = api_key

    def fetch_range(self, symbol=SYMBOL, interval="5min",
                    start_date=None, end_date=None, outputsize=5000):
        params = {
            'symbol': symbol, 'interval': interval,
            'apikey': self.api_key, 'format': 'JSON',
            'order': 'ASC', 'outputsize': outputsize
        }
        if start_date: params['start_date'] = start_date
        if end_date:   params['end_date']   = end_date

        # ★ v4.2 FIX 4: no retry on 4xx (permanent client errors)
        for attempt in range(5):
            try:
                r = requests.get(self.BASE_URL, params=params, timeout=30)
                # ★ v4.2: check 4xx BEFORE raise_for_status so we can skip cleanly
                if 400 <= r.status_code < 500:
                    if r.status_code == 429:
                        logging.warning("429 rate limit — انتظار 60 ث..."); time.sleep(60); continue
                    # 4xx (other than 429) = permanent error — لا فائدة من الـ retry
                    logging.info(f"ℹ️ {r.status_code} ({interval}) — البيانات غير متاحة لهذه الفترة (لا retry)")
                    return pd.DataFrame()
                r.raise_for_status()
                data = r.json()
                if 'code' in data and data['code'] == 429:
                    logging.warning("429 — انتظار 60 ث..."); time.sleep(60); continue
                if 'values' not in data:
                    msg = data.get('message','unknown')
                    if "No data" in msg or "not available" in msg.lower():
                        logging.info(f"ℹ️ نهاية البيانات ({interval})")
                    else:
                        logging.error(f"❌ API error ({interval}): {msg}")
                    return pd.DataFrame()

                df = pd.DataFrame(data['values'])
                if df.empty: return df
                if 'volume' not in df.columns: df['volume'] = 0
                for col in ['open','high','low','close','volume']:
                    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
                df['datetime'] = pd.to_datetime(df['datetime'])
                df.sort_values('datetime', inplace=True)
                df.reset_index(drop=True, inplace=True)
                return df
            except requests.exceptions.Timeout as e:
                logging.warning(f"⏱ Timeout محاولة {attempt+1}/5 ({interval})")
                if attempt == 4: break
                time.sleep((2**attempt) + random.random())
            except requests.exceptions.ConnectionError as e:
                logging.warning(f"🔌 Connection error محاولة {attempt+1}/5")
                if attempt == 4: break
                time.sleep((2**attempt) + random.random())
            except requests.exceptions.HTTPError as e:
                # 5xx server errors — retry
                if e.response is not None and e.response.status_code >= 500:
                    logging.warning(f"5xx server error محاولة {attempt+1}/5")
                    if attempt == 4: break
                    time.sleep((2**attempt) + random.random())
                else:
                    logging.error(f"HTTP error: {e}"); break
            except Exception as e:
                logging.error(f"خطأ: {e}"); break
        return pd.DataFrame()


def fetch_full_history(target_candles=100000, interval="5min", max_requests=25):
    fetcher  = TwelveDataFetcher(TWELVE_DATA_API_KEY)
    all_data = []
    end_date = datetime.utcnow()
    batch    = 5000

    if "min" in interval:
        mult = int(interval.replace("min",""))*60
    elif interval.endswith("h"):
        mult = int(interval.replace("h",""))*3600
    elif "d" in interval:
        mult = 86400
    elif "week" in interval:
        mult = 604800
    else:
        mult = 60

    for _ in range(max_requests):
        start = end_date - timedelta(seconds=batch*mult)
        df_b  = fetcher.fetch_range(
            symbol=SYMBOL, interval=interval,
            start_date=start.strftime("%Y-%m-%d %H:%M:%S"),
            end_date=end_date.strftime("%Y-%m-%d %H:%M:%S"),
            outputsize=batch
        )
        if df_b.empty: break
        all_data.append(df_b)
        logging.info(f"تجميع ~{len(all_data)*batch:,} شمعة ({interval})")
        end_date = df_b['datetime'].min() - timedelta(seconds=1)
        time.sleep(0.6)
        if len(all_data)*batch >= target_candles: break

    if not all_data: return pd.DataFrame()
    df = pd.concat(all_data, ignore_index=True)
    df.drop_duplicates(subset='datetime', keep='first', inplace=True)
    df.sort_values('datetime', inplace=True)
    df.reset_index(drop=True, inplace=True)
    if len(df) > target_candles: df = df.iloc[-target_candles:]
    csv_name = f"{SYMBOL_SAFE}_{interval}_raw.csv"
    df.to_csv(csv_name, index=False)
    logging.info(f"✅ حُفظ {len(df):,} شمعة → {csv_name}")
    return df


def get_or_update_data(symbol_safe, interval="5min",
                       target_candles=100000, force_refresh=False):
    csv_path = f"{symbol_safe}_{interval}_raw.csv"
    if os.path.exists(csv_path) and not force_refresh:
        logging.info(f"✅ ملف موجود: {csv_path} — تحديث ...")
        df_old = pd.read_csv(csv_path)
        df_old['datetime'] = pd.to_datetime(df_old['datetime'])
        last_date = df_old['datetime'].max()
        fetcher   = TwelveDataFetcher(TWELVE_DATA_API_KEY)
        df_new    = fetcher.fetch_range(
            symbol=SYMBOL, interval=interval,
            start_date=(last_date+timedelta(seconds=1)).strftime("%Y-%m-%d %H:%M:%S"),
            end_date=datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
            outputsize=1000)
        if df_new.empty: return df_old
        df = pd.concat([df_old, df_new], ignore_index=True)
        df.drop_duplicates(subset='datetime', keep='last', inplace=True)
        df.sort_values('datetime', inplace=True)
        df.to_csv(csv_path, index=False)
        return df

    max_req = 60 if interval in ["1h","4h","1d"] else 25
    return fetch_full_history(target_candles=target_candles,
                              interval=interval, max_requests=max_req)
print("✅ Data fetcher ready")

✅ Data fetcher ready


##  Feature Engineer — مع Features خاصة بـ SOL

**الطبقات:**
1. Time features (session strength معدّل لـ SOL — US أقوى)
2. Technical indicators (EMA, RSI, MACD, ATR, BB, CCI, ADX...)
3. Volume features (CVD, VPT, MFI, funding_proxy)
4. Price action (candles patterns, wicks, engulfing)
5. Trend filter (EMA alignment, slopes)
6. Market structure (HH/HL/LH/LL, BOS, swings)
7. Support & Resistance
8. VWAP + OBV
9. Digital indicators (Ichimoku, KAMA, ZLEMA, CMO, fractal)
10. **★ SOL-SPECIFIC features** (جديدة كليّاً — انفجار، v-recovery، vol shock...)
11. HTF alignment score

In [14]:
# ══════════════════════════════════════════════════════════════
# FEATURE ENGINEER — مع Features خاصة بـ SOL
# ══════════════════════════════════════════════════════════════
class FeatureEngineer:
    def __init__(self):
        self.scaler          = RobustScaler()
        self.feature_columns = None
        self.scaler_fitted   = False
        self.selector        = None

    # ── 1. Time (SOL: US session dominance أوضح من ETH) ────
    def add_time_features(self, df):
        df = df.copy()
        df['hour']        = df['datetime'].dt.hour
        df['day_of_week'] = df['datetime'].dt.dayofweek
        df['month']       = df['datetime'].dt.month
        df['hour_sin']    = np.sin(2*np.pi*df['hour']/24)
        df['hour_cos']    = np.cos(2*np.pi*df['hour']/24)
        df['dow_sin']     = np.sin(2*np.pi*df['day_of_week']/7)
        df['dow_cos']     = np.cos(2*np.pi*df['day_of_week']/7)
        h = df['hour']
        df['is_asia']    = ((h>=0)&(h<9)).astype(int)
        df['is_europe']  = ((h>=7)&(h<16)).astype(int)
        df['is_us']      = ((h>=13)&(h<22)).astype(int)
        df['is_overlap'] = ((h>=13)&(h<16)).astype(int)
        df['is_weekend'] = (df['day_of_week']>=5).astype(int)
        df['is_monday']  = (df['day_of_week']==0).astype(int)
        df['is_high_vol_hour'] = ((h>=13)&(h<22)).astype(int)
        # ★ SOL-specific: US session أقوى من ETH (retail-driven)
        df['us_session_strength'] = (
            df['is_us'] * 3.5 + df['is_overlap'] * 4.0
            + df['is_europe'] * 1.3 + df['is_asia'] * 0.7
        )
        df['session_momentum'] = (
            df['is_us']*3.0 + df['is_europe']*1.6
            + df['is_overlap']*3.5 + df['is_asia']*0.9
            - df['is_weekend']*0.6
        )
        # SOL bagholders نشطين على الويكند (memecoins)
        df['weekend_activity'] = df['is_weekend'] * 1.2
        return df

    # ── 2. Technical Indicators ─────────────────────────────
    def compute_indicators(self, df):
        df = df.copy()
        c,h,l,o,v = df['close'],df['high'],df['low'],df['open'],df['volume']
        n = len(df)
        p14 = min(14, max(3, n//6)); p20 = min(20, max(4, n//5))
        p50 = min(50, max(5, n//4)); p200 = min(200, max(10, n//2))

        df['ema_9']   = c.ewm(span=9,adjust=False).mean()
        df['ema_21']  = c.ewm(span=21,adjust=False).mean()
        df['ema_50']  = c.ewm(span=p50,adjust=False).mean()
        df['ema_100'] = c.ewm(span=min(100,max(8,n//3)),adjust=False).mean()
        df['ema_200'] = c.ewm(span=p200,adjust=False).mean()
        df['sma_20']  = c.rolling(p20).mean()
        df['sma_50']  = c.rolling(p50).mean()

        delta = c.diff()
        gain  = delta.clip(lower=0).rolling(p14).mean()
        loss  = (-delta.clip(upper=0)).rolling(p14).mean()
        df['rsi']       = 100 - (100/(1 + gain/(loss+1e-9)))
        rsi_min = df['rsi'].rolling(p14).min()
        rsi_max = df['rsi'].rolling(p14).max()
        df['stoch_rsi'] = (df['rsi']-rsi_min)/(rsi_max-rsi_min+1e-9)

        ema12 = c.ewm(span=min(12,max(3,n//8)),adjust=False).mean()
        ema26 = c.ewm(span=min(26,max(5,n//5)),adjust=False).mean()
        macd = ema12 - ema26
        sig  = macd.ewm(span=min(9,max(3,n//10)),adjust=False).mean()
        df['macd']        = macd
        df['macd_signal'] = sig
        df['macd_hist']   = macd - sig

        tr = pd.concat([h-l,(h-c.shift(1)).abs(),(l-c.shift(1)).abs()],axis=1).max(axis=1)
        df['atr']     = tr.rolling(p14).mean()
        df['atr_pct'] = df['atr']/(c+1e-9)

        bb_mid = c.rolling(p20).mean(); bb_std = c.rolling(p20).std()
        df['bb_upper'] = bb_mid + 2*bb_std
        df['bb_lower'] = bb_mid - 2*bb_std
        df['bb_width'] = (df['bb_upper']-df['bb_lower'])/(bb_mid+1e-9)
        df['bb_pct']   = (c-df['bb_lower'])/(df['bb_upper']-df['bb_lower']+1e-9)

        tp = (h+l+c)/3
        df['cci'] = (tp - tp.rolling(p20).mean())/(0.015*tp.rolling(p20).std()+1e-9)
        hh14 = h.rolling(p14).max(); ll14 = l.rolling(p14).min()
        df['williams_r'] = -100*(hh14-c)/(hh14-ll14+1e-9)

        pos_mf = (tp*v).where(tp>tp.shift(1),0).rolling(p14).sum()
        neg_mf = (tp*v).where(tp<tp.shift(1),0).rolling(p14).sum()
        df['mfi'] = 100 - (100/(1 + pos_mf/(neg_mf+1e-9)))

        plus_dm  = (h-h.shift(1)).clip(lower=0)
        minus_dm = (l.shift(1)-l).clip(lower=0)
        plus_dm  = plus_dm.where(plus_dm>minus_dm,0)
        minus_dm = minus_dm.where(minus_dm>plus_dm,0)
        atr14 = tr.rolling(p14).mean()
        pdi = 100*plus_dm.rolling(p14).mean()/(atr14+1e-9)
        mdi = 100*minus_dm.rolling(p14).mean()/(atr14+1e-9)
        dx  = (pdi-mdi).abs()/(pdi+mdi+1e-9)*100
        df['adx']=dx.rolling(p14).mean(); df['adx_plus']=pdi; df['adx_minus']=mdi

        s5 = min(5,max(1,n//15)); s10 = min(10,max(2,n//10))
        df['roc_5']  = c.pct_change(s5)
        df['roc_10'] = c.pct_change(s10)

        body = (c-o).abs(); range_ = (h-l).clip(lower=1e-9)
        df['body_ratio'] = body/range_
        df['upper_wick'] = (h-c.clip(lower=o))/range_
        df['lower_wick'] = (c.clip(upper=o)-l)/range_
        df['candle_dir'] = np.sign(c-o)
        df['vol_regime'] = c.pct_change().rolling(p20).std()
        df['vol_ratio']  = df['atr']/(df['atr'].rolling(p50).mean()+1e-9)
        return df

    # ── 3. Volume ────────────────────────────────────────────
    def add_advanced_volume(self, df):
        df = df.copy()
        c,v,h,l = df['close'],df['volume'],df['high'],df['low']
        df['volume_delta']   = v * np.sign(c.diff())
        df['cvd']            = df['volume_delta'].cumsum()
        df['cvd_slope']      = df['cvd'].diff(5)/(df['cvd'].abs()+1e-9)
        df['cvd_divergence'] = np.where(df['cvd_slope']*c.pct_change()<0,1,0)
        df['vpt']            = (v*c.pct_change()).cumsum()
        df['vpt_slope']      = df['vpt'].diff(5)/(df['vpt'].abs()+1e-9)
        vol_ma20 = v.rolling(20).mean()
        df['vol_ratio_20'] = v/(vol_ma20+1e-9)
        # ★ SOL: عتبة أعلى للـ surge (SOL volume أكثر تقلباً)
        df['volume_surge'] = (v>vol_ma20*3.0).astype(int)  # 3x بدلاً من 2.5x
        bull_vol = v.where(c>c.shift(1),0); bear_vol = v.where(c<c.shift(1),0)
        df['vol_imbalance'] = (bull_vol.rolling(10).sum()-bear_vol.rolling(10).sum())/(v.rolling(10).sum()+1e-9)
        price_up = (c.diff()>0).astype(int); vol_up = (v.diff()>0).astype(int)
        df['volume_divergence'] = np.where(price_up!=vol_up,1,0)
        df['eom']     = (((h+l)/2 - (h.shift(1)+l.shift(1))/2)/(v/(h-l+1e-9)+1e-9))
        df['eom_sma'] = df['eom'].rolling(14).mean()
        # ★ SOL: funding proxy
        df['funding_proxy'] = (c.pct_change().rolling(10).mean()
                              * (1/(df['vol_ratio_20']+0.5)))
        return df

    # ── 4. Price Action ─────────────────────────────────────
    def add_price_action(self, df):
        df = df.copy()
        o,h,l,c = df['open'],df['high'],df['low'],df['close']
        body = (c-o).abs(); range_ = (h-l+1e-9)
        upper_wick = h-c.clip(lower=o); lower_wick = c.clip(upper=o)-l
        df['pinbar_bull']    = ((lower_wick/range_>0.6)&(body/range_<0.3)).astype(int)
        df['pinbar_bear']    = ((upper_wick/range_>0.6)&(body/range_<0.3)).astype(int)
        df['engulfing_bull'] = ((c.shift(1)<o.shift(1))&(c>o)&(o<=c.shift(1))&(c>=o.shift(1))).astype(int)
        df['engulfing_bear'] = ((c.shift(1)>o.shift(1))&(c<o)&(o>=c.shift(1))&(c<=o.shift(1))).astype(int)
        df['inside_bar']  = ((h<h.shift(1))&(l>l.shift(1))).astype(int)
        df['outside_bar'] = ((h>h.shift(1))&(l<l.shift(1))).astype(int)
        df['doji']        = (body/range_<0.1).astype(int)
        df['three_white_soldiers'] = ((c>o)&(c.shift(1)>o.shift(1))&(c.shift(2)>o.shift(2))
                                      &(c>c.shift(1))&(c.shift(1)>c.shift(2))).astype(int)
        df['three_black_crows'] = ((c<o)&(c.shift(1)<o.shift(1))&(c.shift(2)<o.shift(2))
                                   &(c<c.shift(1))&(c.shift(1)<c.shift(2))).astype(int)
        df['gap_up']   = (o>h.shift(1)).astype(int)
        df['gap_down'] = (o<l.shift(1)).astype(int)
        df['price_accel'] = c.diff().diff()
        rr = h.rolling(20).max() - l.rolling(20).min()
        df['close_to_range_ratio'] = (c - l.rolling(20).min())/(rr+1e-9)
        return df

    # ── 5. Trend Filter ─────────────────────────────────────
    def add_trend_filter(self, df):
        df = df.copy(); c = df['close']
        df['trend_filter']     = (df['ema_50']>df['ema_200']).astype(int)
        df['trend_filter_9']   = (df['ema_9']>df['ema_21']).astype(int)
        df['ema_slope_50']     = df['ema_50'].diff(3)/(c+1e-9)
        df['ema_slope_200']    = df['ema_200'].diff(3)/(c+1e-9)
        df['price_vs_ema50']   = (c>df['ema_50']).astype(int)
        df['price_vs_ema200']  = (c>df['ema_200']).astype(int)
        df['price_vs_ema9']    = (c>df['ema_9']).astype(int)
        df['trend_strength']   = (df['ema_50']-df['ema_200'])/(c+1e-9)
        df['trend_strength_9'] = (df['ema_9']-df['ema_21'])/(c+1e-9)
        df['ema_alignment'] = (
            (df['ema_9']>df['ema_21']).astype(int)
            + (df['ema_21']>df['ema_50']).astype(int)
            + (df['ema_50']>df['ema_100']).astype(int)
            + (df['ema_100']>df['ema_200']).astype(int)
        )/4.0
        return df

    # ── 6. Market Structure ────────────────────────────────
    def add_market_structure(self, df, window=10):
        df = df.copy()
        high,low,c = df['high'],df['low'],df['close']
        rh = high.rolling(window).max(); rl = low.rolling(window).min()
        prh = rh.shift(window); prl = rl.shift(window)
        df['hh'] = (rh>prh).astype(int); df['hl'] = (rl>prl).astype(int)
        df['lh'] = (rh<prh).astype(int); df['ll'] = (rl<prl).astype(int)
        df['market_structure'] = np.where((df['hh']==1)&(df['hl']==1),1,
                                          np.where((df['lh']==1)&(df['ll']==1),-1,0))
        df['bos_up']   = (c>rh.shift(1)).astype(int)
        df['bos_down'] = (c<rl.shift(1)).astype(int)
        df['swing_high'] = ((high>high.shift(1))&(high>high.shift(-1))
                           &(high>high.shift(2))&(high>high.shift(-2))).astype(int)
        df['swing_low']  = ((low<low.shift(1))&(low<low.shift(-1))
                           &(low<low.shift(2))&(low<low.shift(-2))).astype(int)
        return df

    # ── 7. Support & Resistance ────────────────────────────
    def add_support_resistance(self, df, window=20):
        df = df.copy(); c = df['close']
        w   = min(window, max(5, len(df)//6))
        w50 = min(50, max(5, len(df)//4))
        df['resistance'] = df['high'].rolling(w).max()
        df['support']    = df['low'].rolling(w).min()
        df['dist_to_resistance'] = (df['resistance']-c)/(c+1e-9)
        df['dist_to_support']    = (c-df['support'])/(c+1e-9)
        df['near_resistance']    = (df['dist_to_resistance']<0.005).astype(int)
        df['near_support']       = (df['dist_to_support']<0.005).astype(int)
        sr_range = df['resistance']-df['support']
        df['sr_dist'] = (c-df['support'])/(sr_range+1e-9)
        df['resistance_50'] = df['high'].rolling(w50).max()
        df['support_50']    = df['low'].rolling(w50).min()
        df['sr_dist_50']    = (c-df['support_50'])/(df['resistance_50']-df['support_50']+1e-9)
        return df

    # ── 8. VWAP + OBV ──────────────────────────────────────
    def add_volume_profile(self, df):
        df = df.copy()
        c,h,l,v = df['close'],df['high'],df['low'],df['volume']
        tp = (h+l+c)/3
        vw = min(144, max(5, len(df)//4))
        ow = min(20, max(3, len(df)//10))
        vol_sum = v.rolling(vw).sum()+1e-9
        df['vwap'] = (tp*v).rolling(vw).sum()/vol_sum
        df['dist_to_vwap']  = (c-df['vwap'])/(df['vwap']+1e-9)
        df['price_vs_vwap'] = (c>df['vwap']).astype(int)
        vwap_std = (tp-df['vwap']).rolling(vw).std()
        df['vwap_upper2'] = df['vwap']+2.0*vwap_std
        df['vwap_lower2'] = df['vwap']-2.0*vwap_std
        df['vwap_band_pct'] = (c-df['vwap_lower2'])/(df['vwap_upper2']-df['vwap_lower2']+1e-9)
        obv_dir = np.sign(c.diff())
        df['obv']        = (v*obv_dir).cumsum()
        df['obv_sma']    = df['obv'].rolling(ow).mean()
        df['obv_slope']  = df['obv'].diff(min(5,len(df)//8))/(df['obv'].abs()+1e-9)
        df['obv_vs_sma'] = (df['obv']>df['obv_sma']).astype(int)
        return df

    # ── 9. Digital Indicators ──────────────────────────────
    def add_digital_indicators(self, df):
        df = df.copy(); c,h,l = df['close'],df['high'],df['low']; n=len(df)
        t9=min(9,max(3,n//8)); t26=min(26,max(5,n//5)); t52=min(52,max(8,n//3))
        sh26 = min(26,max(3,n//6))
        tenkan   = (h.rolling(t9).max()+l.rolling(t9).min())/2
        kijun    = (h.rolling(t26).max()+l.rolling(t26).min())/2
        senkou_a = ((tenkan+kijun)/2).shift(sh26)
        senkou_b = ((h.rolling(t52).max()+l.rolling(t52).min())/2).shift(sh26)
        df['ichi_tenkan']         = tenkan
        df['ichi_kijun']          = kijun
        df['ichi_senkou_a']       = senkou_a
        df['ichi_senkou_b']       = senkou_b
        df['ichi_cloud_bull']     = (senkou_a>senkou_b).astype(int)
        df['ichi_above_cloud']    = ((c>senkou_a)&(c>senkou_b)).astype(int)
        df['ichi_tk_cross']       = ((tenkan>kijun)&(tenkan.shift(1)<=kijun.shift(1))).astype(int)
        df['ichi_price_vs_kijun'] = (c>kijun).astype(int)

        cmo_p = min(14,max(3,n//6))
        d_ = c.diff()
        up_ = d_.clip(lower=0).rolling(cmo_p).sum()
        dn_ = (-d_.clip(upper=0)).rolling(cmo_p).sum()
        df['cmo'] = 100*(up_-dn_)/(up_+dn_+1e-9)

        er_p = min(10,max(3,n//8))
        fast_sc = 2/3; slow_sc = 2/31
        direction  = c.diff(er_p).abs()
        volatility = c.diff().abs().rolling(er_p).sum()
        er = direction/(volatility+1e-9)
        sc = (er*(fast_sc-slow_sc)+slow_sc)**2
        kama_arr = c.to_numpy(dtype=float).copy()
        for i in range(er_p, len(kama_arr)):
            kama_arr[i] = kama_arr[i-1] + sc.iloc[i]*(kama_arr[i]-kama_arr[i-1])
        df['kama']          = kama_arr
        df['price_vs_kama'] = (c>df['kama']).astype(int)
        kslope_p = min(3,max(1,n//12))
        df['kama_slope'] = pd.Series(kama_arr,index=c.index).diff(kslope_p)/(c+1e-9)

        zl_p = min(21,max(5,n//5)); lag_ = (zl_p-1)//2
        zl_input = c + (c-c.shift(lag_))
        df['zlema'] = zl_input.ewm(span=zl_p,adjust=False).mean()
        df['price_vs_zlema'] = (c>df['zlema']).astype(int)
        zslope_p = min(3,max(1,n//12))
        df['zlema_slope'] = df['zlema'].diff(zslope_p)/(c+1e-9)

        fd_n = min(20,max(4,n//5))
        df['fractal_dim'] = (h.rolling(fd_n).max()-l.rolling(fd_n).min())/(df['atr']*np.sqrt(fd_n)+1e-9)
        return df

    # ── 10. ★★★ SOL-SPECIFIC FEATURES ★★★ ─────────────────
    def add_sol_specific(self, df):
        """
        Features مصممة خصيصاً لسلوك SOL/USD:
        - explosive moves detection (SOL شهير بحركات > 3σ)
        - V-shape recovery patterns (SOL بيرتد سريع بعد الشلالات)
        - vol shock (قفزات مفاجئة في الحجم = whale activity)
        - extended trend runs (SOL بيمشي في الترند لمدة أطول من ETH)
        - downside momentum (SOL crashes أشد)
        - range expansion signals (SOL بينفجر بعد الركود)
        - bull/bear pressure (قوة الشراء/البيع)
        - session-specific SOL behavior
        """
        df = df.copy()
        c, h, l, o, v = df['close'], df['high'], df['low'], df['open'], df['volume']
        ret = c.pct_change()

        # 1) Volatility Regime (نفس ETH لكن معايَر)
        vol_20 = ret.rolling(20).std()
        vol_60 = ret.rolling(60).std()
        df['vol_regime_htf'] = vol_20 / (vol_60 + 1e-9)
        df['vol_regime_shift'] = df['vol_regime_htf'].diff(5).fillna(0)

        # 2) Volatility of Volatility
        df['vol_of_vol'] = vol_20.rolling(20).std()
        df['vol_of_vol_ratio'] = df['vol_of_vol'] / (df['vol_of_vol'].rolling(50).mean()+1e-9)

        # 3) ★ SOL Explosive Move Detector — حركة > 3σ في شمعة واحدة
        ret_std_20 = ret.rolling(20).std()
        ret_zscore = ret / (ret_std_20 + 1e-9)
        df['explosive_move_up'] = (ret_zscore > 3.0).astype(int)
        df['explosive_move_dn'] = (ret_zscore < -3.0).astype(int)
        df['explosive_intensity'] = ret_zscore.clip(-5, 5) / 5.0
        # explosive في آخر 5 شموع (memory)
        df['recent_explosion_up'] = df['explosive_move_up'].rolling(5).sum()
        df['recent_explosion_dn'] = df['explosive_move_dn'].rolling(5).sum()

        # 4) ★ Breakout Intensity (SOL يحب البريك أوت)
        rolling_high_20 = h.rolling(20).max().shift(1)
        rolling_low_20  = l.rolling(20).min().shift(1)
        rng_20 = rolling_high_20 - rolling_low_20
        df['breakout_intensity_up'] = np.where(
            c > rolling_high_20,
            (c - rolling_high_20) / (rng_20 + 1e-9), 0)
        df['breakout_intensity_dn'] = np.where(
            c < rolling_low_20,
            (rolling_low_20 - c) / (rng_20 + 1e-9), 0)
        df['breakout_signal'] = (
            df['breakout_intensity_up'] - df['breakout_intensity_dn']
        )

        # 5) ★ V-Shape Recovery Detector — علامة تجارية لـ SOL
        # يقيس: انخفاض قوي ثم ارتداد قوي في نافذة قصيرة
        low_5 = l.rolling(5).min()
        drop_pct = (c.shift(5) - low_5) / (c.shift(5) + 1e-9)
        recovery_pct = (c - low_5) / (low_5 + 1e-9)
        df['v_recovery_signal'] = np.where(
            (drop_pct > 0.03) & (recovery_pct > 0.02) & (c > c.shift(3)),
            recovery_pct, 0)
        df['v_recovery_strength'] = df['v_recovery_signal'].rolling(3).max()

        # 6) ★ Vol Shock — قفزات مفاجئة (whale activity)
        vol_ma_10 = v.rolling(10).mean()
        vol_std_10 = v.rolling(10).std()
        vol_z = (v - vol_ma_10) / (vol_std_10 + 1e-9)
        df['vol_shock'] = (vol_z > 2.5).astype(int)
        df['vol_shock_intensity'] = vol_z.clip(-3, 5) / 5.0
        df['recent_vol_shock'] = df['vol_shock'].rolling(10).sum()

        # 7) ★ Extended Trend Runs — SOL بيمشي في الترند طويل
        ema20 = c.ewm(span=20, adjust=False).mean()
        above_ema = (c > ema20).astype(int)
        # طول الرن الحالي فوق/تحت EMA20
        streak_change = (above_ema != above_ema.shift(1)).cumsum()
        df['trend_run_length'] = above_ema.groupby(streak_change).cumcount()
        df['trend_run_normalized'] = (df['trend_run_length'] / 20.0).clip(0, 3)

        # 8) ★ Downside Momentum — SOL crashes أشد من ETH
        neg_ret = ret.where(ret < 0, 0)
        pos_ret = ret.where(ret > 0, 0)
        df['downside_momentum'] = neg_ret.rolling(10).sum()
        df['upside_momentum']   = pos_ret.rolling(10).sum()
        df['momentum_asymmetry'] = df['upside_momentum'] + df['downside_momentum']

        # 9) ★ Range Compression → Expansion (SOL bread & butter)
        rng = h - l
        rng_ma_10 = rng.rolling(10).mean()
        rng_ma_50 = rng.rolling(50).mean()
        df['range_compression'] = rng_ma_10 / (rng_ma_50 + 1e-9)
        df['range_expansion_signal'] = (
            (df['range_compression'].shift(3) < 0.6).astype(int)
            & (df['range_compression'] > 1.1).astype(int)
        ).astype(int)
        # ratio لكشف الاستعداد للانفجار
        df['compression_ready'] = (df['range_compression'] < 0.7).astype(int)

        # 10) ★ Bull/Bear Pressure — قوة الشراء/البيع مقاسة بالجسم × الحجم
        body_signed = (c - o) / (c + 1e-9)
        df['bull_pressure'] = (body_signed.clip(lower=0) * v).rolling(10).sum() / (v.rolling(10).sum() + 1e-9)
        df['bear_pressure'] = (body_signed.clip(upper=0).abs() * v).rolling(10).sum() / (v.rolling(10).sum() + 1e-9)
        df['pressure_ratio'] = df['bull_pressure'] - df['bear_pressure']

        # 11) Momentum Persistence
        up_count = (ret > 0).rolling(10).sum()
        dn_count = (ret < 0).rolling(10).sum()
        df['momentum_persistence'] = (up_count - dn_count) / 10.0

        # 12) HH/LL intensity
        df['hh_intensity'] = (h - h.shift(1)).clip(lower=0).rolling(5).sum() / (df['atr']+1e-9)
        df['ll_intensity'] = (l.shift(1) - l).clip(lower=0).rolling(5).sum() / (df['atr']+1e-9)

        # 13) Consecutive candles direction streak
        sign_c = np.sign(c - c.shift(1))
        streak = (sign_c != sign_c.shift(1)).cumsum()
        df['direction_streak'] = sign_c * sign_c.groupby(streak).cumcount().add(1)

        # 14) Close position in candle
        rng_c = (h - l).clip(lower=1e-9)
        df['close_pos_in_candle'] = (c - l) / rng_c

        # 15) Rolling Sharpe (momentum quality)
        df['rolling_sharpe_10'] = (
            ret.rolling(10).mean() / (ret.rolling(10).std() + 1e-9)
        )

        # 16) ATR-normalized returns
        df['atr_norm_return_5']  = c.pct_change(5)  / (df['atr_pct']+1e-9)
        df['atr_norm_return_10'] = c.pct_change(10) / (df['atr_pct']+1e-9)

        # 17) ★ SOL: Distance from all-time high (rolling)
        # SOL بيبقى حساس أوي لمستوى الـ ATH التاريخي
        ath_rolling = c.rolling(500, min_periods=50).max()
        df['dist_from_ath'] = (c - ath_rolling) / (ath_rolling + 1e-9)

        # 18) ★ SOL: acceleration (2nd derivative of returns)
        df['return_accel'] = ret.diff().rolling(5).mean()

        return df

    # ── 11. HTF Alignment + Digital Score (SOL-tuned weights) ─
    def add_htf_alignment(self, df):
        df  = df.copy(); idx = df.index
        _g  = lambda col: df[col] if col in df.columns else pd.Series(0.0,index=idx)
        # SOL: وزن أعلى لـ momentum_persistence و explosive
        df['confirmation_score'] = (
            0.20*_g('trend_filter')
            + 0.16*_g('market_structure').clip(-1,1)*0.5
            + 0.15*_g('cvd_slope').clip(-1,1)
            + 0.06*(1-_g('volume_divergence'))
            + 0.12*(_g('adx')/100).clip(0,1)
            + 0.08*_g('ema_alignment')
            + 0.05*_g('bos_up')
            + 0.05*_g('ichi_above_cloud')
            + 0.07*_g('momentum_persistence').clip(-1,1)  # ★ SOL: زيادة
            + 0.06*_g('pressure_ratio').clip(-1,1)         # ★ SOL: جديد
        )
        df['signal_strength'] = (
            df['confirmation_score']*0.30
            + _g('trend_strength').clip(-1,1)*0.20
            + _g('vol_imbalance').clip(-1,1)*0.15
            + _g('breakout_signal').clip(-1,1)*0.20      # ★ SOL
            + _g('v_recovery_strength').clip(0,0.2)*0.5 * 0.15  # ★ SOL جديد
        )
        df['digital_score'] = (
            _g('ichi_above_cloud')*0.25
            + _g('price_vs_kama')*0.20
            + _g('price_vs_zlema')*0.15
            + (_g('cmo')/100).clip(-1,1)*0.10
            + (1-_g('fractal_dim').clip(0,2)/2)*0.08
            + _g('rolling_sharpe_10').clip(-2,2)/2*0.10
            + _g('trend_run_normalized').clip(0,3)/3*0.12  # ★ SOL جديد
        )
        return df

    # ── MAIN ───────────────────────────────────────────────
    def create_features(self, df, shift=True):
        df = self.add_time_features(df)
        df = self.compute_indicators(df)
        df = self.add_advanced_volume(df)
        df = self.add_price_action(df)
        df = self.add_trend_filter(df)
        df = self.add_market_structure(df)
        df = self.add_support_resistance(df)
        df = self.add_volume_profile(df)
        df = self.add_digital_indicators(df)
        df = self.add_sol_specific(df)          # ★ SOL-specific (replaces ETH)
        df = self.add_htf_alignment(df)

        if shift:
            excl = {'datetime','open','high','low','close','volume'}
            cols = [c for c in df.columns if c not in excl]
            df[cols] = df[cols].shift(1)

        df = df.ffill().fillna(0)
        df.replace([np.inf,-np.inf],0,inplace=True)
        warmup = max(10, min(60, len(df)//10))
        df = df.iloc[warmup:].copy()

        thresh = int(len(df)*0.3)
        df = df.dropna(axis=1, thresh=thresh).ffill().fillna(0)
        df.replace([np.inf,-np.inf],0,inplace=True)

        excl_f = {'datetime','open','high','low','close','volume',
                  'future_close','pct_change','target',
                  'hit_tp1','hit_tp2','hit_tp3','hit_tp4','hit_sl'}
        self.feature_columns = [c for c in df.columns if c not in excl_f]
        return df

    def create_target(self, df, tf_config):
        df = df.copy()
        horizon = tf_config.get('horizon',10)
        sl_m  = tf_config.get('atr_mult_sl',3.0)
        tp1_m = tf_config.get('atr_mult_tp1',1.5)
        tp2_m = tf_config.get('atr_mult_tp2',2.5)
        tp3_m = tf_config.get('atr_mult_tp3',3.5)
        tp4_m = tf_config.get('atr_mult_tp4',5.0)
        closes = df['close'].values; atrs = df['atr'].values
        highs  = df['high'].values;  lows = df['low'].values
        n = len(df)
        hit_sl  = np.zeros(n,dtype=int); hit_tp1 = np.zeros(n,dtype=int)
        hit_tp2 = np.zeros(n,dtype=int); hit_tp3 = np.zeros(n,dtype=int)
        hit_tp4 = np.zeros(n,dtype=int); target  = np.zeros(n,dtype=int)
        for i in range(n-horizon):
            entry = closes[i]; atr = max(atrs[i],1e-6)
            sl  = entry - sl_m*atr
            tp1 = entry + tp1_m*atr; tp2 = entry + tp2_m*atr
            tp3 = entry + tp3_m*atr; tp4 = entry + tp4_m*atr
            for j in range(i+1, min(i+horizon+1, n)):
                lo = lows[j]; hi = highs[j]
                if lo <= sl:
                    hit_sl[i] = 1; break
                if hi >= tp1:
                    hit_tp1[i] = 1; target[i] = 1
                if hi >= tp2: hit_tp2[i] = 1
                if hi >= tp3: hit_tp3[i] = 1
                if hi >= tp4: hit_tp4[i] = 1
        df['hit_sl']=hit_sl; df['hit_tp1']=hit_tp1; df['hit_tp2']=hit_tp2
        df['hit_tp3']=hit_tp3; df['hit_tp4']=hit_tp4; df['target']=target.astype(int)
        df = df.iloc[:-horizon].copy()
        return df

    def select_features(self, X, y, max_features=80):
        k = min(max_features, X.shape[1])
        if k == X.shape[1]:
            self.selector = None
            return X
        self.selector = SelectKBest(f_classif, k=k)
        X_sel = self.selector.fit_transform(X, y)
        selected = X.columns[self.selector.get_support()]
        self.feature_columns = list(selected)
        return pd.DataFrame(X_sel, index=X.index, columns=selected)

    def apply_feature_selection(self, X):
        if self.selector is None: return X
        if isinstance(X, pd.DataFrame): X = X[self.feature_columns]
        return pd.DataFrame(self.selector.transform(X),
                            index=X.index if hasattr(X,'index') else None,
                            columns=self.feature_columns)

    def fit_scaler(self, X):
        self.scaler.fit(X); self.scaler_fitted = True
        if isinstance(X, pd.DataFrame):
            self.feature_columns = list(X.columns)

    def transform(self, X):
        if not self.scaler_fitted:
            raise RuntimeError("يجب تدريب الـ scaler أولاً")
        if isinstance(X, pd.DataFrame):
            for col in self.feature_columns:
                if col not in X.columns: X[col] = 0.0
            X = X[self.feature_columns]
        return self.scaler.transform(X)

    def fit_transform(self, X):
        self.fit_scaler(X); return self.transform(X)


class MultiTimeframeBuilder:
    def __init__(self): self._htf_cache = {}
    def _build_htf_df(self, df_raw, resample_rule, tf_config=None):
        if resample_rule in self._htf_cache: return self._htf_cache[resample_rule]
        df_htf = (df_raw.set_index('datetime').resample(resample_rule)
                  .agg({'open':'first','high':'max','low':'min','close':'last','volume':'sum'})
                  .dropna().reset_index())
        fe = FeatureEngineer()
        df_htf_f = fe.create_features(df_htf, shift=True)
        df_htf_f = fe.create_target(df_htf_f, tf_config=tf_config or {})
        self._htf_cache[resample_rule] = df_htf_f
        return df_htf_f
    def merge_htf_features(self, df_ltf, df_raw, htf_resample_rule,
                           htf_feature_names, htf_label, tf_config=None):
        if not htf_feature_names or htf_resample_rule is None: return df_ltf
        df_htf = self._build_htf_df(df_raw, htf_resample_rule, tf_config)
        available = [c for c in htf_feature_names if c in df_htf.columns]
        if not available: return df_ltf
        df_sel = df_htf[['datetime']+available].copy()
        df_sel.columns = ['datetime'] + [f'htf_{htf_label}_{c}' for c in available]
        df_merged = pd.merge_asof(df_ltf.sort_values('datetime'),
                                  df_sel.sort_values('datetime'),
                                  on='datetime', direction='backward').ffill().fillna(0)
        return df_merged

print("✅ Feature Engineer with SOL-specific features loaded")

✅ Feature Engineer with SOL-specific features loaded


##  Backtester — SOL-Optimized

**التحسينات:**
- Kelly-lite confidence sizing (حجم متناسب مع ثقة الموديل)
- ATR trailing stop (ديناميكي بعد TP1)  
- Daily loss cap
- 4 tranches TP (TP1/TP2/TP3/TP4)
- Break-even move بعد TP1
- Move-to-TP1 بعد TP2

In [16]:
# ══════════════════════════════════════════════════════════════
# BACKTESTER — SOL-Optimized
# ✅ Kelly-lite confidence sizing
# ✅ ATR trailing stop
# ✅ Daily loss cap
# ✅ 4-tranches TP with progressive SL tightening
# ══════════════════════════════════════════════════════════════
class Backtester:
    def __init__(self):
        self.initial_capital = INITIAL_CAPITAL

    def _exec(self, price, is_buy):
        sp = SPREAD_BPS/10000
        if is_buy: return price*(1+sp)*(1+SLIPPAGE)
        return price*(1-sp)*(1-SLIPPAGE)

    def _size_from_confidence(self, prob, buy_th):
        if not USE_CONFIDENCE_SIZING:
            return 1.0
        excess = max(0.0, prob - buy_th) / max(1e-6, 1.0 - buy_th)
        factor = MIN_SIZE_FRACTION + (MAX_SIZE_FRACTION - MIN_SIZE_FRACTION) * excess
        return float(np.clip(factor, MIN_SIZE_FRACTION, MAX_SIZE_FRACTION))

    def run(self, df, model, fe, feature_columns, tf_config,
            use_fold_thresholds=False, custom_buy_th=None):

        capital = float(self.initial_capital)
        start_cap = capital
        positions = []; trades = []; equity = [capital]
        sl_m  = tf_config.get('atr_mult_sl',3.0)
        tp1_m = tf_config.get('atr_mult_tp1',1.5)
        tp2_m = tf_config.get('atr_mult_tp2',2.5)
        tp3_m = tf_config.get('atr_mult_tp3',3.5)
        tp4_m = tf_config.get('atr_mult_tp4',5.0)

        key_buy = 'fold_buy_threshold' if use_fold_thresholds else 'buy_threshold'
        buy_th  = custom_buy_th if custom_buy_th is not None else tf_config.get(key_buy, 0.55)

        max_daily = tf_config.get('max_trades_per_day')
        min_adx   = tf_config.get('min_adx', 0)

        # ★ v4.3 NEW: signal-quality gate parameters
        min_prob_margin  = tf_config.get('min_prob_margin', MIN_PROB_MARGIN)
        min_adx_strict   = tf_config.get('min_adx_strict', 0)
        max_trades_fold  = tf_config.get('max_trades_per_fold')
        horizon_bars     = tf_config.get('horizon', 10)

        # State for circuit breaker + trade cap
        consecutive_losses = 0
        cooldown_until     = -1
        fold_trade_count   = 0

        daily_counts = {}; daily_loss = {}

        missing = [c for c in feature_columns+['close','atr'] if c not in df.columns]
        if missing:
            logging.error(f"أعمدة مفقودة: {missing}"); return {}

        X_scaled = fe.transform(df[feature_columns].copy())
        all_proba = model.predict_proba(X_scaled)[:,1]
        closes = df['close'].values; highs = df['high'].values
        lows   = df['low'].values;   atrs = df['atr'].values
        adxs   = df['adx'].values if 'adx' in df.columns else np.full(len(df), 25.0)
        # ★ v4.3: squeeze detector (BB inside KC = ranging/tight market)
        if 'bb_width' in df.columns and 'kc_width' in df.columns:
            bbw = df['bb_width'].values
            kcw = df['kc_width'].values
            squeezes = (bbw < kcw * 0.85).astype(int)  # BB much narrower than KC
        elif 'bb_width' in df.columns:
            # fallback: use rolling percentile of bb_width
            bbw = df['bb_width'].values
            bbw_ser = pd.Series(bbw).rolling(20, min_periods=5).quantile(0.25).values
            squeezes = (bbw < bbw_ser).astype(int)
        else:
            squeezes = np.zeros(len(df), dtype=int)
        dts    = df['datetime'].values if 'datetime' in df.columns else [None]*len(df)

        for i in range(len(df)):
            p   = float(closes[i]); hi = float(highs[i]); lo = float(lows[i])
            atr = max(float(atrs[i]),1e-6); prob = float(all_proba[i])
            dk  = str(dts[i])[:10] if dts[i] is not None else "unknown"

            # ── فحص الصفقات المفتوحة ────────────────────
            remaining = []
            for pos in positions:
                tp_levels = [
                    pos['entry'] + tp1_m*pos['atr'],
                    pos['entry'] + tp2_m*pos['atr'],
                    pos['entry'] + tp3_m*pos['atr'],
                    pos['entry'] + tp4_m*pos['atr'],
                ]
                # Trailing SL: SOL يحتاج ATR trail أوسع (2.5x بدلاً 2x)
                if pos.get('tp2_hit'):
                    pos['sl'] = max(pos['sl'], tp_levels[0])
                elif pos.get('tp1_hit') and pos['sl'] < pos['entry']:
                    pos['sl'] = pos['entry']
                if pos.get('tp1_hit'):
                    atr_trail = p - 2.5 * atr  # ⬆️ من 2.0 (SOL يحتاج مسافة أوسع)
                    pos['sl'] = max(pos['sl'], atr_trail)

                if lo <= pos['sl']:
                    total_pos_pnl = 0.0
                    for k,(units_k,done_k) in enumerate(pos['tranches']):
                        if not done_k and units_k > 0:
                            ep  = self._exec(pos['sl'], False)
                            pnl = units_k*(ep - pos['entry'])
                            capital += units_k*ep
                            total_pos_pnl += pnl
                            daily_loss[dk] = daily_loss.get(dk,0.0) + max(0.0,-pnl/start_cap)
                            trades.append({'profit':pnl,'exit_reason':'stop_loss','tp_level':0,
                                          'return_pct':pnl/start_cap*100})
                    # ★ v4.3: track consecutive losses (aggregate position PnL)
                    if total_pos_pnl < 0:
                        consecutive_losses += 1
                        if consecutive_losses >= MAX_CONSECUTIVE_LOSSES:
                            cooldown_until = i + int(horizon_bars * COOLDOWN_BARS_MULT)
                            consecutive_losses = 0   # reset after triggering
                    else:
                        consecutive_losses = 0
                    continue

                new_tranches = []
                for k,(units_k,done_k) in enumerate(pos['tranches']):
                    if done_k or units_k == 0:
                        new_tranches.append((units_k,done_k)); continue
                    if hi >= tp_levels[k]:
                        ep = self._exec(tp_levels[k], False)
                        pnl = units_k*(ep - pos['entry'])
                        capital += units_k*ep
                        if k==0: pos['tp1_hit'] = True
                        if k==1: pos['tp2_hit'] = True
                        trades.append({'profit':pnl,'exit_reason':'take_profit','tp_level':k+1,
                                      'return_pct':pnl/start_cap*100})
                        new_tranches.append((units_k,True))
                        consecutive_losses = 0   # ★ v4.3: reset streak on any TP hit
                    else:
                        new_tranches.append((units_k,False))
                pos['tranches'] = new_tranches
                if any(not d for u_,d in new_tranches if u_>0):
                    remaining.append(pos)
            positions = remaining

            # ── إشارة جديدة ─────────────────────────────
            signal = "HOLD"

            # ★ v4.3 QUALITY GATE 1: probability margin
            # لا نتاجر لو prob بس فوق buy_th بشعرة — لازم margin حقيقي
            if prob > buy_th + min_prob_margin: signal = "BUY"

            # ADX regime filter (existing)
            if signal == "BUY" and min_adx > 0:
                if adxs[i] < min_adx: signal = "HOLD"

            # ★ v4.3 QUALITY GATE 2: strict ADX filter (higher than min_adx)
            # Only enter in CLEAR trending markets
            if signal == "BUY" and min_adx_strict > 0:
                if adxs[i] < min_adx_strict: signal = "HOLD"

            # ★ v4.3 QUALITY GATE 3: squeeze filter
            # Skip trades when BB is inside KC (ranging market — costs dominate)
            if signal == "BUY" and USE_SQUEEZE_FILTER and squeezes[i] > 0:
                signal = "HOLD"

            # ★ v4.3 QUALITY GATE 4: cost-aware filter
            # TP1 gain must be >= MIN_COST_COVERAGE × round-trip costs
            if signal == "BUY":
                tp1_gain_pct = (tp1_m * atr) / p
                round_trip_cost = (SPREAD_BPS / 10000) * 2 + SLIPPAGE * 2 + 0.00075 * 2
                if tp1_gain_pct < round_trip_cost * MIN_COST_COVERAGE:
                    signal = "HOLD"

            # ★ v4.3 QUALITY GATE 5: circuit breaker (consecutive losses)
            if signal == "BUY" and i < cooldown_until:
                signal = "HOLD"

            # ★ v4.3 QUALITY GATE 6: max trades per fold cap
            if signal == "BUY" and max_trades_fold is not None:
                if fold_trade_count >= max_trades_fold:
                    signal = "HOLD"

            # existing daily filters
            if signal == "BUY" and max_daily:
                if daily_counts.get(dk,0) >= max_daily: signal = "HOLD"
            if signal == "BUY":
                if daily_loss.get(dk,0.0) >= MAX_DAILY_RISK_PCT: signal = "HOLD"

            if signal == "BUY" and not positions:
                ep = self._exec(p, True)
                conf_factor = self._size_from_confidence(prob, buy_th)

                # ★ v4.2 FIX 1: TRUE risk-based sizing (was: notional-based, gave ~14x too small)
                #   risk_dollars = max $ we can lose if SL hits
                #   sl_dist      = $ per unit lost if SL hits
                #   units        = risk_dollars / sl_dist
                risk_dollars = capital * RISK_PCT_PER_TRADE * conf_factor
                sl_dist_per_unit = sl_m * atr   # SL distance in $/unit
                if sl_dist_per_unit <= 1e-9:
                    equity.append(capital); continue   # invalid SL — skip

                total_units = risk_dollars / sl_dist_per_unit
                notional    = total_units * ep

                # Cap notional to prevent extreme leverage on low-vol candles
                # (e.g. tiny ATR on ranging bars would size huge without cap)
                max_notional = capital * MAX_POSITION_PCT
                if notional > max_notional:
                    total_units = max_notional / ep
                    notional    = total_units * ep

                # Also require enough free capital
                if notional > capital * 0.98:
                    equity.append(capital); continue   # not enough capital

                # ★ v4.2 POLISH 1: front-loaded tranche weights [0.40, 0.30, 0.20, 0.10]
                # Old was [0.25]*4 → TP4 rarely hit, average win was tiny
                tranche_weights = [0.40, 0.30, 0.20, 0.10]
                tranches = [(total_units * w, False) for w in tranche_weights]

                positions.append({
                    'entry': ep,
                    'sl':    ep - sl_m*atr,
                    'atr':   atr,
                    'tp1_hit': False, 'tp2_hit': False,
                    'tranches': tranches,
                    'notional_at_entry': notional,
                    'risk_at_entry':    risk_dollars,
                })
                capital -= notional
                daily_counts[dk] = daily_counts.get(dk,0) + 1
                fold_trade_count += 1   # ★ v4.3: track for max_trades_per_fold cap

            mtm = sum(units_k*p for pos in positions
                     for units_k,done_k in pos['tranches']
                     if not done_k and units_k>0)
            equity.append(capital + mtm)

        # إغلاق المتبقي
        for pos in positions:
            for k,(units_k,done_k) in enumerate(pos['tranches']):
                if not done_k and units_k>0:
                    ep  = self._exec(closes[-1], False)
                    pnl = units_k*(ep - pos['entry'])
                    capital += units_k*ep
                    trades.append({'profit':pnl,'exit_reason':'end_of_data','tp_level':k+1,
                                  'return_pct':pnl/start_cap*100})

        if not trades: return {}

        eq = pd.Series(equity)
        wins = [t for t in trades if t['profit']>0]
        loss = [t for t in trades if t['profit']<=0]
        wr = len(wins)/len(trades)*100
        tp = sum(t['profit'] for t in wins)
        tl = abs(sum(t['profit'] for t in loss))
        dd = ((eq - eq.cummax())/(eq.cummax()+1e-9)*100).min()
        aw = tp/len(wins) if wins else 0
        al = tl/len(loss) if loss else 0
        final_capital = equity[-1]
        total_ret_pct = (final_capital - self.initial_capital)/self.initial_capital*100
        tp_hits = {k: sum(1 for t in trades if t.get('tp_level')==k) for k in [1,2,3,4]}
        sl_hits = sum(1 for t in trades if t['exit_reason']=='stop_loss')

        eq_ret = eq.pct_change().dropna()
        sharpe = (eq_ret.mean()/(eq_ret.std()+1e-9))*np.sqrt(252) if len(eq_ret)>5 else 0

        # ★ v4.3: removed buggy SANITY WARNING (each position = 4 tranche exits,
        #         so trade count includes tranche closings, plus trailing stops
        #         intentionally lock in tiny gains). The old check was noise.
        # Meaningful check: warn if actual avg loss is FAR from configured risk
        avg_loss_pct = (al / self.initial_capital * 100) if al else 0

        return {
            'total_trades': len(trades),
            'win_rate': round(wr,2),
            'profit_factor': round(tp/tl,2) if tl>0 else float('inf'),
            'total_return_pct': round(total_ret_pct,2),
            'final_capital': round(final_capital,2),
            'max_drawdown_pct': round(dd,2),
            'avg_win_usd': round(aw,2),
            'avg_loss_usd': round(al,2),
            'avg_loss_pct_of_capital': round(avg_loss_pct, 3),  # ★ v4.2: verify sizing works
            'expectancy_usd': round((wr/100*aw) - ((1-wr/100)*al),2) if al else 0,
            'sharpe_ratio': round(sharpe,2),
            'tp1_hits': tp_hits[1], 'tp2_hits': tp_hits[2],
            'tp3_hits': tp_hits[3], 'tp4_hits': tp_hits[4],
            'sl_hits': sl_hits,
            'sl_rate': round(sl_hits/len(trades)*100,1),
        }

print("✅ Backtester v4.3 — signal quality gates + cost-aware filter + circuit breaker")

✅ Backtester v4.3 — signal quality gates + cost-aware filter + circuit breaker


##  Threshold Optimizer + Confidence Metrics

- **Threshold Optimizer**: يبحث تلقائياً عن أفضل عتبة (buy_threshold) اللي تعطي أعلى profit factor
- **Confidence Report**: Bootstrap 95% CI + Reliability Grade (A/B/C/D)

In [17]:
# ══════════════════════════════════════════════════════════════
# ★ v4.2 FIX 2: helper to split test set into VAL (for threshold opt)
#              and TEST (for honest evaluation) — prevents data leakage
# ══════════════════════════════════════════════════════════════
def split_val_test(df_test, val_ratio=0.5):
    """Split a test fold into VAL (first half) for threshold optimization
    and TEST (last half) for honest out-of-sample evaluation.

    Prevents the data leakage bug where the same window was used for
    both optimizing buy_threshold AND measuring the resulting PF —
    which inflated holdout PFs to 75.94, 16.25, 14.6 etc in v4.1.
    """
    if len(df_test) < 20:
        # too small to split — return all as both val and test (leakage but small impact)
        return df_test.copy(), df_test.copy()
    split = int(len(df_test) * val_ratio)
    return df_test.iloc[:split].copy(), df_test.iloc[split:].copy()


# ══════════════════════════════════════════════════════════════
# THRESHOLD OPTIMIZER (SOL-tuned scoring)
# ★ v4.2: caller MUST pass a VAL slice, NOT the test slice
#         (see split_val_test above)
# ══════════════════════════════════════════════════════════════
def optimize_threshold(df_val, model, fe, feature_columns, tf_config,
                       min_trades=5, threshold_range=(0.45, 0.78), step=0.01):
    """SOL: threshold_range أوسع للأعلى (0.78) لأن SOL يحتاج فلترة أشد أحياناً."""
    best_th = tf_config.get('buy_threshold', 0.55)
    best_score = -np.inf
    best_metrics = {}

    ths = np.arange(threshold_range[0], threshold_range[1] + 1e-9, step)
    all_results = []

    for th in ths:
        bt = Backtester()
        m = bt.run(df_val, model, fe, feature_columns, tf_config,
                   custom_buy_th=float(th))
        if not m or m['total_trades'] < min_trades:
            continue
        pf_raw = m.get('profit_factor', 0)
        pf = min(pf_raw, 5.0)
        pf_bonus = max(0, min(pf - 2.0, 3.0)) * 0.15
        wr = m.get('win_rate', 0) / 100
        dd = abs(m.get('max_drawdown_pct', 0)) / 100
        n_trades = m['total_trades']
        trade_bonus = min(1.0, n_trades / (min_trades * 3))
        trade_penalty = 0.0 if n_trades >= min_trades*1.5 else 0.10
        # SOL: ندعم Sharpe أكتر (تقلب أعلى)
        sharpe = m.get('sharpe_ratio', 0)
        sharpe_bonus = min(0.15, max(0, sharpe) * 0.05)
        score = (pf * 0.42 + pf_bonus + wr * 0.20 + trade_bonus * 0.15
                 + sharpe_bonus - dd * 0.12 - trade_penalty)
        all_results.append({'th': float(th), 'score': score, **m})
        if score > best_score:
            best_score = score
            best_th = float(th)
            best_metrics = m

    return best_th, best_metrics, all_results


# ══════════════════════════════════════════════════════════════
# CONFIDENCE METRICS
# ══════════════════════════════════════════════════════════════
def bootstrap_ci(values, n_boot=1000, ci=95):
    if len(values) < 3:
        return (float(np.mean(values)) if len(values) else 0, 0, 0)
    vals = np.array(values)
    boots = [np.mean(np.random.choice(vals, size=len(vals), replace=True))
             for _ in range(n_boot)]
    lo = np.percentile(boots, (100-ci)/2)
    hi = np.percentile(boots, 100-(100-ci)/2)
    return float(np.mean(vals)), float(lo), float(hi)


def compute_confidence_report(fold_metrics, holdout_metrics, timeframe_config):
    valid_folds = [f for f in fold_metrics if f.get('backtest')]
    if not valid_folds:
        return {'grade': 'F', 'reason': 'no valid folds',
                'consistency_score': 0, 'n_profitable_folds': 0, 'n_folds': 0,
                'total_trades_walkforward': 0,
                'profit_factor_mean': 0, 'profit_factor_95ci': [0,0],
                'return_mean_pct': 0, 'return_95ci_pct': [0,0],
                'win_rate_mean': 0, 'win_rate_95ci': [0,0],
                'holdout_profit_factor': 0, 'holdout_trades': 0,
                'holdout_return_pct': 0}

    pfs = [f['backtest'].get('profit_factor', 0) for f in valid_folds]
    pfs = [p if p != float('inf') else 5.0 for p in pfs]
    rets = [f['backtest'].get('total_return_pct', 0) for f in valid_folds]
    wrs  = [f['backtest'].get('win_rate', 0) for f in valid_folds]
    trades_per_fold = [f['backtest'].get('total_trades', 0) for f in valid_folds]

    n_profitable = sum(1 for p in pfs if p > 1.0)
    consistency = n_profitable / len(valid_folds)

    pf_mean, pf_lo, pf_hi = bootstrap_ci(pfs)
    ret_mean, ret_lo, ret_hi = bootstrap_ci(rets)
    wr_mean, wr_lo, wr_hi = bootstrap_ci(wrs)

    total_trades_walk = sum(trades_per_fold)
    holdout_pf = holdout_metrics.get('profit_factor', 0) if holdout_metrics else 0
    holdout_trades = holdout_metrics.get('total_trades', 0) if holdout_metrics else 0
    holdout_ret = holdout_metrics.get('total_return_pct', 0) if holdout_metrics else 0

    # Reliability Grade (نفس ETH — معايير موحدة)
    grade = 'D'; reason = []
    if consistency >= 0.80 and pf_lo >= 1.30 and total_trades_walk >= 15 and holdout_ret > 0:
        grade = 'A'; reason.append("consistent & strong across folds & holdout")
    elif consistency >= 0.65 and pf_lo >= 1.10 and total_trades_walk >= 10:
        grade = 'B'; reason.append("solid consistency, decent CI")
    elif consistency >= 0.50 and pf_mean >= 1.10:
        grade = 'C'; reason.append("moderate — usable but risky")
    else:
        grade = 'D'; reason.append("low consistency / low PF")

    # ★ v4.4: WF vs Holdout consistency check
    wf_holdout_mismatch = None
    if pf_mean < 1.2 and holdout_pf > 3.0 and holdout_trades >= 5:
        wf_holdout_mismatch = (
            f"⚠️ WF PF ({pf_mean:.2f}) << Holdout PF ({holdout_pf:.2f}) "
            f"— consider disabling fold_threshold_opt"
        )

    return {
        'grade': grade,
        'reason': "; ".join(reason),
        'consistency_score': round(consistency, 3),
        'n_profitable_folds': n_profitable,
        'n_folds': len(valid_folds),
        'total_trades_walkforward': total_trades_walk,
        'profit_factor_mean': round(pf_mean, 2),
        'profit_factor_95ci': [round(pf_lo, 2), round(pf_hi, 2)],
        'return_mean_pct': round(ret_mean, 2),
        'return_95ci_pct': [round(ret_lo, 2), round(ret_hi, 2)],
        'win_rate_mean': round(wr_mean, 2),
        'win_rate_95ci': [round(wr_lo, 2), round(wr_hi, 2)],
        'holdout_profit_factor': holdout_pf,
        'holdout_trades': holdout_trades,
        'holdout_return_pct': holdout_ret,
        'wf_holdout_mismatch': wf_holdout_mismatch,   # ★ v4.4
    }

print("✅ Threshold optimizer + confidence metrics ready")

✅ Threshold optimizer + confidence metrics ready


##  Walk-Forward Training

**التدفق:**
1. Resample data → timeframe
2. Create features + target
3. Feature selection (SelectKBest)
4. TimeSeriesSplit (5 folds) مع Embargo
5. Train XGBoost لكل fold
6. Optimize threshold على validation set
7. Choose best fold model
8. Train Final Model على كل البيانات
9. Holdout Backtest (آخر 20%)
10. Confidence Report + Save

In [18]:
# ══════════════════════════════════════════════════════════════
# SINGLE FOLD FALLBACK
# ══════════════════════════════════════════════════════════════
def _train_single_fold(X, y, xgb_param, timeframe, max_features):
    neg = (y==0).sum(); pos = (y==1).sum()
    spw = neg/pos if pos>0 else 1.0
    fe = FeatureEngineer()
    X  = fe.select_features(X, y, max_features=max_features)
    fe.fit_scaler(X); Xs = fe.transform(X)
    mdl = xgb.XGBClassifier(
        objective='binary:logistic', n_estimators=300,
        scale_pos_weight=spw, random_state=42,
        tree_method='hist', eval_metric='logloss',
        verbosity=0, **xgb_param
    )
    mdl.fit(Xs, y, verbose=False)
    logging.info(f"[{timeframe}] ✅ Single-fold — {len(X)} عينة")
    return mdl, fe


# ══════════════════════════════════════════════════════════════
# WALK-FORWARD TRAINING
# ══════════════════════════════════════════════════════════════
def train_model_with_walk_forward(df_raw, timeframe, n_splits=5, mtf_builder=None):
    config = TIMEFRAME_CONFIG[timeframe]
    resample_rule = config['resample_rule']
    years = config['years']
    min_candles = config.get('min_candles', 100)
    larger_tf = config.get('larger_tf')
    htf_features = config.get('htf_features', [])
    min_trades_fold = config.get('min_trades_fold', 2)
    max_features = MAX_FEATURES.get(timeframe, 70)

    logging.info("=" * 65)
    logging.info(f"Walk-Forward [SOL/USD v4] — فريم {timeframe}")
    logging.info("=" * 65)

    df_tf = (df_raw.set_index('datetime').resample(resample_rule)
             .agg({'open':'first','high':'max','low':'min','close':'last','volume':'sum'})
             .dropna().reset_index())
    if 'datetime' not in df_tf.columns:
        df_tf.rename(columns={'index':'datetime'}, inplace=True)

    # ★ v4.4 FIX 3: cap 1W years at FORCE_MAX_YEARS_1W (SOL doesn't exist before 2020)
    if timeframe == '1W' and years and years > FORCE_MAX_YEARS_1W:
        logging.warning(f"[1W] years={years} capped to {FORCE_MAX_YEARS_1W} (SOL launched 2020)")
        years = FORCE_MAX_YEARS_1W

    if years:
        cutoff = df_tf['datetime'].max() - timedelta(days=365*years)
        df_tf  = df_tf[df_tf['datetime'] >= cutoff].copy()

    logging.info(f"شموع بعد القطع ({years}y): {len(df_tf)}")
    if len(df_tf) < min_candles:
        logging.error(f"❌ بيانات غير كافية: {len(df_tf)} < {min_candles}")
        return None

    fe = FeatureEngineer()
    df_feat   = fe.create_features(df_tf, shift=True)
    df_target = fe.create_target(df_feat, tf_config=config)

    if df_target.empty or len(df_target) < min_candles:
        logging.error(f"❌ بيانات ناقصة بعد features: {len(df_target)}")
        return None

    # HTF merge
    if mtf_builder and larger_tf and htf_features:
        htf_cfg = TIMEFRAME_CONFIG.get(larger_tf)
        if htf_cfg:
            logging.info(f"إضافة HTF features من {larger_tf} ...")
            df_target = mtf_builder.merge_htf_features(
                df_ltf=df_target, df_raw=df_raw,
                htf_resample_rule=htf_cfg['resample_rule'],
                htf_feature_names=htf_features,
                htf_label=larger_tf, tf_config=htf_cfg)

    excl_f = {'datetime','open','high','low','close','volume',
              'future_close','pct_change','target',
              'hit_tp1','hit_tp2','hit_tp3','hit_tp4','hit_sl'}
    raw_features = [c for c in df_target.columns if c not in excl_f]
    X_raw = df_target[raw_features]
    y = df_target['target']

    fe_sel = FeatureEngineer()
    X = fe_sel.select_features(X_raw, y, max_features=max_features)
    selected_features = fe_sel.feature_columns

    tp1_rate = df_target['hit_tp1'].mean() if 'hit_tp1' in df_target else 0
    tp4_rate = df_target['hit_tp4'].mean() if 'hit_tp4' in df_target else 0
    sl_rate  = df_target['hit_sl'].mean() if 'hit_sl' in df_target else 0
    logging.info(f"Features: {len(selected_features)} | "
                 f"TP1:{tp1_rate:.1%} TP4:{tp4_rate:.1%} SL:{sl_rate:.1%}")

    pos_ratio = y.mean()
    if pos_ratio < 0.20 or pos_ratio > 0.80:
        logging.warning(f"⚠️ [{timeframe}] class imbalance: pos={pos_ratio:.1%}")

    xgb_param = XGB_PARAMS.get(timeframe, XGB_PARAMS["default"])

    if len(X) < 100:
        min_train = max(15, len(X)//4); min_test = max(5, len(X)//10)
        n_splits = min(n_splits, 3)
    else:
        min_train = max(30, len(X)//(n_splits+1))
        min_test  = max(8, len(X)//(n_splits*3))

    effective_splits = min(n_splits, max(2, len(X)//(min_train+min_test)))
    tscv = TimeSeriesSplit(n_splits=effective_splits)

    embargo = config.get('horizon', 10)

    fold_metrics = []
    best_model = None; best_fe_fold = None
    best_score = -np.inf; best_threshold = config.get('buy_threshold', 0.55)

    for fold, (tr_idx, te_idx) in enumerate(tscv.split(X)):
        if len(tr_idx) > embargo:
            tr_idx = tr_idx[:-embargo]

        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

        if len(X_tr) < min_train or len(X_te) < min_test:
            logging.warning(f"Fold {fold+1}: train={len(X_tr)} test={len(X_te)} — تخطي")
            continue

        logging.info(f"── Fold {fold+1}/{effective_splits} (train={len(X_tr)}, test={len(X_te)})")

        neg = (y_tr==0).sum(); pos = (y_tr==1).sum()
        spw = neg/pos if pos>0 else 1.0
        # ★ v4.3: SMOTE-like boost for imbalanced TFs (esp. 1W with ~10% pos)
        smote_mult = config.get('smote_multiplier', 1.0)
        if smote_mult > 1.0:
            spw = spw * smote_mult
            logging.info(f"  [{timeframe}] ★ SMOTE-like boost: spw ×{smote_mult} = {spw:.2f}")

        fe_fold = FeatureEngineer()
        fe_fold.feature_columns = selected_features
        fe_fold.fit_scaler(X_tr)
        X_tr_s = fe_fold.transform(X_tr); X_te_s = fe_fold.transform(X_te)

        model = xgb.XGBClassifier(
            objective='binary:logistic', n_estimators=600,
            early_stopping_rounds=50, scale_pos_weight=spw,
            random_state=42, tree_method='hist',
            eval_metric='logloss', verbosity=0, **xgb_param)

        # ★ v4.4 FIX 2: Recency-weighted training (SOL evolves fast — memecoin era ≠ 2021 DeFi)
        if USE_RECENCY_WEIGHTING and len(X_tr) > 10:
            sample_w = np.linspace(RECENCY_WEIGHT_MIN, RECENCY_WEIGHT_MAX, len(X_tr))
            model.fit(X_tr_s, y_tr, sample_weight=sample_w,
                      eval_set=[(X_te_s, y_te)], verbose=False)
        else:
            model.fit(X_tr_s, y_tr, eval_set=[(X_te_s, y_te)], verbose=False)

        y_pred = model.predict(X_te_s)
        acc = accuracy_score(y_te, y_pred)
        precision = precision_score(y_te, y_pred, zero_division=0)
        recall = recall_score(y_te, y_pred, zero_division=0)

        df_test = df_target.iloc[te_idx].copy()

        # ★ v4.4 FIX 1: DISABLE per-fold threshold optimizer
        # Root cause of noisy WF folds: optimize_threshold on tiny val slice (~8% data)
        # picks lucky thresholds → too many trades → noisy PF. Now use fold_buy_threshold
        # directly on FULL df_test slice → cleaner, matches holdout style.
        bt = Backtester()
        if DISABLE_FOLD_THRESHOLD_OPT:
            metrics_test = bt.run(df_test, model, fe_fold, selected_features, config,
                                  use_fold_thresholds=True)
            used_th = config.get('fold_buy_threshold', 0.5)
        else:
            # legacy path (kept for A/B testing)
            df_val, df_test_eval = split_val_test(df_test, val_ratio=0.5)
            opt_th, opt_metrics_on_val, _ = optimize_threshold(
                df_val, model, fe_fold, selected_features, config,
                min_trades=max(2, min_trades_fold // 2))
            metrics_orig = bt.run(df_test_eval, model, fe_fold, selected_features, config,
                                  use_fold_thresholds=True)
            metrics_opt = bt.run(df_test_eval, model, fe_fold, selected_features, config,
                                 custom_buy_th=opt_th) if opt_metrics_on_val else metrics_orig
            pick_opt = (opt_metrics_on_val and metrics_opt and
                        metrics_opt.get('profit_factor',0) > metrics_orig.get('profit_factor',0))
            metrics_test = metrics_opt if pick_opt else metrics_orig
            used_th = opt_th if pick_opt else config.get('fold_buy_threshold', 0.5)

        # ★ v4.2 FIX 5: cap PF at 5.0 to prevent outlier folds from inflating averages
        if metrics_test and metrics_test.get('profit_factor', 0) > 5.0:
            metrics_test['profit_factor_raw'] = metrics_test['profit_factor']
            metrics_test['profit_factor'] = 5.0

        logging.info(f"  Fold {fold+1} — Acc:{acc:.2%} Prec:{precision:.2%} Rec:{recall:.2%}")
        if metrics_test:
            logging.info(f"  BT — WR:{metrics_test['win_rate']}% "
                        f"PF:{metrics_test['profit_factor']} "
                        f"Ret:{metrics_test['total_return_pct']}% "
                        f"SL:{metrics_test['sl_rate']}% "
                        f"Trades:{metrics_test['total_trades']} "
                        f"(th={used_th:.2f})")
        else:
            logging.info("  BT — لا صفقات في هذا الفولد")

        fold_metrics.append({
            'fold': fold+1, 'accuracy': acc,
            'precision': precision, 'recall': recall,
            'backtest': metrics_test, 'optimized_threshold': used_th
        })

        if metrics_test and metrics_test['total_trades'] >= min_trades_fold:
            sl_pen = metrics_test['sl_rate']/100
            score = (metrics_test.get('profit_factor',0)*0.45
                    + metrics_test.get('win_rate',0)*0.25
                    + acc*0.10 - sl_pen*0.20)
        else:
            score = acc * 8

        if score > best_score:
            best_score = score
            best_model = model
            best_fe_fold = fe_fold
            best_threshold = used_th

    if best_model is None:
        logging.warning(f"⚠️ [{timeframe}] Single Fold Fallback ...")
        best_model, best_fe_fold = _train_single_fold(
            X, y, xgb_param, timeframe, max_features)
        best_fe_fold.feature_columns = selected_features

    # Final Model — كامل البيانات
    neg_f = (y==0).sum(); pos_f = (y==1).sum()
    n_est = 600
    if hasattr(best_model, 'best_iteration') and best_model.best_iteration:
        n_est = max(80, best_model.best_iteration + 1)

    fe_full = FeatureEngineer()
    fe_full.feature_columns = selected_features
    fe_full.fit_scaler(X); X_full_s = fe_full.transform(X)

    # ★ v4.3: apply SMOTE-like boost for imbalanced TFs
    final_spw = neg_f/pos_f if pos_f else 1
    smote_mult_final = config.get('smote_multiplier', 1.0)
    if smote_mult_final > 1.0:
        final_spw = final_spw * smote_mult_final

    final_model = xgb.XGBClassifier(
        objective='binary:logistic', n_estimators=n_est,
        scale_pos_weight=final_spw,
        random_state=42, tree_method='hist',
        eval_metric='logloss', verbosity=0, **xgb_param)

    # ★ v4.4 FIX 2: recency weighting for final model too
    if USE_RECENCY_WEIGHTING and len(X) > 10:
        sample_w_full = np.linspace(RECENCY_WEIGHT_MIN, RECENCY_WEIGHT_MAX, len(X))
        final_model.fit(X_full_s, y, sample_weight=sample_w_full)
    else:
        final_model.fit(X_full_s, y)

    # Holdout Backtest + Threshold Optimization
    holdout_start = int(len(df_target)*0.8)
    df_holdout = df_target.iloc[holdout_start:].copy()
    full_metrics = {}
    final_buy_th = config.get('buy_threshold', 0.55)

    if len(df_holdout) >= 15:
        try:
            # ★ v4.2 FIX 2: split holdout into VAL (threshold opt) + TEST (report)
            df_hold_val, df_hold_test = split_val_test(df_holdout, val_ratio=0.5)

            opt_th_h, opt_m_h_val, _ = optimize_threshold(
                df_hold_val, final_model, fe_full, selected_features,
                config, min_trades=max(2, min_trades_fold // 2))

            bt_h = Backtester()
            # Report metrics on df_hold_test (never seen by optimizer)
            m_default = bt_h.run(df_hold_test, final_model, fe_full,
                                 selected_features, config,
                                 use_fold_thresholds=False)
            m_optimized = bt_h.run(df_hold_test, final_model, fe_full,
                                   selected_features, config,
                                   custom_buy_th=opt_th_h) if opt_m_h_val else m_default

            if opt_m_h_val and m_optimized and \
               m_optimized.get('profit_factor',0) > m_default.get('profit_factor',0):
                full_metrics = m_optimized
                final_buy_th = opt_th_h
            else:
                full_metrics = m_default
                final_buy_th = config.get('buy_threshold', 0.55)

            # ★ v4.4 FIX 3: retry with lower threshold if 0 trades in holdout
            # (fixes 1W getting 0 holdout trades due to class imbalance)
            retry_count = 0
            while (not full_metrics or full_metrics.get('total_trades', 0) == 0) \
                  and retry_count < HOLDOUT_ZERO_TRADE_RETRIES:
                retry_count += 1
                new_th = max(HOLDOUT_RETRY_FLOOR, final_buy_th - HOLDOUT_ZERO_TRADE_STEP)
                logging.warning(f"[{timeframe}] holdout 0 trades — retry {retry_count} "
                                f"with th={new_th:.3f} (was {final_buy_th:.3f})")
                full_metrics = bt_h.run(df_hold_test, final_model, fe_full,
                                        selected_features, config,
                                        custom_buy_th=new_th)
                final_buy_th = new_th
                if full_metrics and full_metrics.get('total_trades', 0) > 0:
                    logging.info(f"[{timeframe}] holdout retry success — {full_metrics['total_trades']} trades")
                    break

            # ★ v4.2 FIX 5: cap PF at 5.0 for holdout too
            if full_metrics and full_metrics.get('profit_factor', 0) > 5.0:
                full_metrics['profit_factor_raw'] = full_metrics['profit_factor']
                full_metrics['profit_factor'] = 5.0
        except Exception as e:
            logging.warning(f"Threshold opt failed: {e}")
            bt_h = Backtester()
            full_metrics = bt_h.run(df_holdout, final_model, fe_full,
                                    selected_features, config,
                                    use_fold_thresholds=False)

        if full_metrics:
            logging.info(f"✅ Holdout [{timeframe}] — "
                        f"WR:{full_metrics['win_rate']}% "
                        f"PF:{full_metrics['profit_factor']} "
                        f"Ret:{full_metrics['total_return_pct']}% "
                        f"Cap:${full_metrics['final_capital']:,.0f} "
                        f"Sharpe:{full_metrics.get('sharpe_ratio',0)} "
                        f"Trades:{full_metrics['total_trades']} "
                        f"(th={final_buy_th:.2f})")

    conf_report = compute_confidence_report(fold_metrics, full_metrics, config)
    logging.info(f"🎯 [{timeframe}] Confidence Grade: {conf_report['grade']} — "
                f"Consistency: {conf_report['consistency_score']*100:.0f}% "
                f"PF CI: {conf_report.get('profit_factor_95ci','-')}")

    # Save
    os.makedirs(MODELS_DIR, exist_ok=True)
    model_path = f"{MODELS_DIR}/{SYMBOL_SAFE}_{timeframe}.pkl"
    config_saved = dict(config)
    config_saved['buy_threshold_optimized'] = final_buy_th
    joblib.dump({
        'model':            final_model,
        'scaler':           fe_full.scaler,
        'feature_columns':  selected_features,
        'config':           config_saved,
        'buy_threshold_optimized': final_buy_th,
        'confidence_grade': conf_report['grade'],
    }, model_path)

    metrics_file = f"{MODELS_DIR}/{SYMBOL_SAFE}_{timeframe}_metrics.json"
    with open(metrics_file, 'w', encoding='utf-8') as f:
        json.dump({
            'symbol': SYMBOL, 'timeframe': timeframe,
            'n_features_selected': len(selected_features),
            'n_features_total': len(raw_features),
            'tp1_rate_train': round(float(tp1_rate),4),
            'tp4_rate_train': round(float(tp4_rate),4),
            'sl_rate_train': round(float(sl_rate),4),
            'fold_results': fold_metrics,
            'holdout_backtest_last_20pct': full_metrics,
            'buy_threshold_optimized': final_buy_th,
            'confidence_report': conf_report,
            'config': config
        }, f, indent=2, default=str, ensure_ascii=False)

    logging.info(f"💾 نموذج محفوظ: {model_path}")
    return final_model, fe_full, conf_report

print("✅ Walk-forward v4.5 — thresholds corrected (lower buy, higher sell)")

✅ Walk-forward v4.5 — thresholds corrected (lower buy, higher sell)


In [19]:
# ══════════════════════════════════════════════════════════════
# TRAIN ALL MODELS — SOL/USD
# ══════════════════════════════════════════════════════════════
def train_all_models(force_refresh_data=False):
    logging.info("=" * 70)
    logging.info("★ SOL/USD — Model Factory v4 (SOL-Optimized) ★")
    logging.info("=" * 70)
    logging.info(f"Risk per trade: {RISK_PCT_PER_TRADE*100:.1f}% | "
                f"Max daily risk: {MAX_DAILY_RISK_PCT*100:.0f}% | "
                f"Capital: ${INITIAL_CAPITAL:,}")

    df_5min = get_or_update_data(SYMBOL_SAFE, interval="5min",
                                  target_candles=100000,
                                  force_refresh=force_refresh_data)
    df_1h   = get_or_update_data(SYMBOL_SAFE, interval="1h",
                                  target_candles=50000,
                                  force_refresh=force_refresh_data)

    mtf_builder = MultiTimeframeBuilder()
    results = {}; confidence_reports = {}

    for tf in TIMEFRAME_CONFIG:
        logging.info("-"*50)
        try:
            source_df = df_1h if tf in ["1h","2h","4h","1d","1W"] else df_5min
            if source_df is None or source_df.empty:
                results[tf] = "❌ بيانات غير موجودة"; continue
            res = train_model_with_walk_forward(
                source_df, tf, n_splits=5, mtf_builder=mtf_builder)
            if res:
                _, _, cr = res
                results[tf] = f"✅ نجح (Grade: {cr['grade']})"
                confidence_reports[tf] = cr
            else:
                results[tf] = "❌ فشل (بيانات غير كافية)"
        except Exception as e:
            logging.error(f"❌ خطأ في {tf}: {e}", exc_info=True)
            results[tf] = f"❌ خطأ: {e}"

    logging.info("=" * 70)
    logging.info("📊 ملخص التدريب النهائي — SOL/USD v4:")
    logging.info("=" * 70)
    logging.info(f"{'TF':<8}{'Status':<28}{'PF (95% CI)':<20}{'Ret %':<12}{'Trades':<10}")
    logging.info("-" * 78)
    for tf, st in results.items():
        cr = confidence_reports.get(tf, {})
        pf_ci = cr.get('profit_factor_95ci', '-')
        ret = cr.get('return_mean_pct', '-')
        trades = cr.get('total_trades_walkforward', '-')
        logging.info(f"{tf:<8}{st:<28}{str(pf_ci):<20}{str(ret):<12}{str(trades):<10}")
    logging.info("=" * 70)
    logging.info(f"📁 الموديلات: {MODELS_DIR}/")

    summary_file = f"{MODELS_DIR}/_summary.json"
    os.makedirs(MODELS_DIR, exist_ok=True)
    with open(summary_file, 'w', encoding='utf-8') as f:
        json.dump({
            'symbol': SYMBOL,
            'trained_at': datetime.utcnow().isoformat(),
            'training_status': results,
            'confidence_reports': confidence_reports,
        }, f, indent=2, default=str, ensure_ascii=False)
    return results, confidence_reports

print("✅ Full training pipeline ready")

✅ Full training pipeline ready


## تشغيل التدريب الكامل

>  **قبل التشغيل**: تأكد إن `TWELVE_DATA_API_KEY` موجود في environment variables.
>
> ```bash
> export TWELVE_DATA_API_KEY="your_key_here"
> ```
>
> أو داخل الـ notebook:
> ```python
> import os
> os.environ["TWELVE_DATA_API_KEY"] = "your_key_here"
> ```

In [10]:
# ══════════════════════════════════════════════════════════════
# تشغيل التدريب الكامل لكل الفريمات (SOL/USD)
# ══════════════════════════════════════════════════════════════

# لو مافيش key معرف، ضعه هنا (اختياري):
# os.environ["TWELVE_DATA_API_KEY"] = "your_api_key"

results, confidence_reports = train_all_models(force_refresh_data=True)

# طباعة تقرير الثقة النهائي
print("\n" + "="*80)
print("🎯 تقرير الثقة النهائي (SOL/USD v4)")
print("="*80)
for tf, cr in confidence_reports.items():
    print(f"\n📊 فريم {tf} — Grade: {cr['grade']}")
    print(f"   Consistency:    {cr['consistency_score']*100:.0f}% "
          f"({cr['n_profitable_folds']}/{cr['n_folds']} folds رابحة)")
    print(f"   Profit Factor:  {cr['profit_factor_mean']} "
          f"(95% CI: {cr['profit_factor_95ci']})")
    print(f"   Return:         {cr['return_mean_pct']}% "
          f"(95% CI: {cr['return_95ci_pct']}%)")
    print(f"   Win Rate:       {cr['win_rate_mean']}% "
          f"(95% CI: {cr['win_rate_95ci']}%)")
    print(f"   Trades (WF):    {cr['total_trades_walkforward']}")
    print(f"   Holdout PF:     {cr['holdout_profit_factor']} "
          f"({cr['holdout_trades']} trades, {cr['holdout_return_pct']}% return)")


🎯 تقرير الثقة النهائي (SOL/USD v4)

📊 فريم 5min — Grade: D
   Consistency:    0% (0/2 folds رابحة)
   Profit Factor:  0.64 (95% CI: [0, 0])
   Return:         -0.22% (95% CI: [0, 0]%)
   Win Rate:       43.75% (95% CI: [0, 0]%)
   Trades (WF):    24
   Holdout PF:     5.0 (4 trades, 0.16% return)

📊 فريم 15min — Grade: D
   Consistency:    0% (0/1 folds رابحة)
   Profit Factor:  0.75 (95% CI: [0, 0])
   Return:         -1.32% (95% CI: [0, 0]%)
   Win Rate:       52.27% (95% CI: [0, 0]%)
   Trades (WF):    88
   Holdout PF:     5.0 (12 trades, 1.4% return)

📊 فريم 45min — Grade: D
   Consistency:    0% (0/4 folds رابحة)
   Profit Factor:  0.68 (95% CI: [0.43, 0.88])
   Return:         -3.05% (95% CI: [-6.66, -0.94]%)
   Win Rate:       39.92% (95% CI: [34.14, 45.7]%)
   Trades (WF):    508
   Holdout PF:     5.0 (40 trades, 3.08% return)

📊 فريم 1h — Grade: D
   Consistency:    0% (0/3 folds رابحة)
   Profit Factor:  0.89 (95% CI: [0.8, 0.96])
   Return:         -2.99% (95% CI: [-4.66,

##  ضغط وحفظ الموديلات

In [20]:
# ══════════════════════════════════════════════════════════════
# تحميل/ضغط الموديلات
# ══════════════════════════════════════════════════════════════
import shutil

MODELS_DIR_ZIP = "models/SOLUSD"
ZIP_NAME       = "SOL_technical.zip"

if not os.path.exists(MODELS_DIR_ZIP):
    print(f"❌ الفولدر '{MODELS_DIR_ZIP}' مش موجود!")
else:
    all_files = os.listdir(MODELS_DIR_ZIP)
    pkl_files  = [f for f in all_files if f.endswith('.pkl')]
    json_files = [f for f in all_files if f.endswith('.json')]
    print(f"📁 محتوى '{MODELS_DIR_ZIP}':")
    print(f"   .pkl  (موديلات): {len(pkl_files)}")
    print(f"   .json (مقاييس):  {len(json_files)}")
    for f in sorted(all_files):
        size = os.path.getsize(os.path.join(MODELS_DIR_ZIP, f))
        print(f"   {f:<50} {size/1024:.1f} KB")

    print(f"\n📦 جاري ضغط الفولدر...")
    shutil.make_archive(ZIP_NAME.replace('.zip',''), 'zip', '.', MODELS_DIR_ZIP)
    size_mb = os.path.getsize(ZIP_NAME)/(1024*1024)
    print(f"✅ تم إنشاء: {ZIP_NAME} ({size_mb:.1f} MB)")

    # Colab: لتحميل الملف
    try:
        from google.colab import files
        print(f"⬇️ جاري التحميل ...")
        files.download(ZIP_NAME)
    except ImportError:
        print(f"ℹ️ لتحميل الملف يدوياً: {ZIP_NAME}")

📁 محتوى 'models/SOLUSD':
   .pkl  (موديلات): 8
   .json (مقاييس):  9
   SOLUSD_15min.pkl                                   147.0 KB
   SOLUSD_15min_metrics.json                          3.4 KB
   SOLUSD_1W.pkl                                      107.7 KB
   SOLUSD_1W_metrics.json                             3.7 KB
   SOLUSD_1d.pkl                                      176.1 KB
   SOLUSD_1d_metrics.json                             3.6 KB
   SOLUSD_1h.pkl                                      900.2 KB
   SOLUSD_1h_metrics.json                             4.5 KB
   SOLUSD_2h.pkl                                      211.6 KB
   SOLUSD_2h_metrics.json                             3.8 KB
   SOLUSD_45min.pkl                                   208.6 KB
   SOLUSD_45min_metrics.json                          4.9 KB
   SOLUSD_4h.pkl                                      261.4 KB
   SOLUSD_4h_metrics.json                             4.3 KB
   SOLUSD_5min.pkl                                    185.9 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>